In [ ]:
import os
import math
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pygame
import imageio.v2 as imageio


# ------------------------------------------------------------
# Headless support (Linux / Colab)
# ------------------------------------------------------------
if os.name != "nt" and os.environ.get("DISPLAY", "") == "":
    os.environ.setdefault("SDL_VIDEODRIVER", "dummy")


# ------------------------------------------------------------
# Weather database
# ------------------------------------------------------------
@dataclass
class WeatherProfile:
    name: str
    speed_factor: float
    braking_factor: float
    visibility_factor: float
    comm_factor: float
    crash_factor: float
    background_color: Tuple[int, int, int]
    overlay_alpha: int


WEATHER_DATABASE: Dict[str, WeatherProfile] = {
    "clear": WeatherProfile("CLEAR", 1.00, 1.00, 1.00, 1.00, 1.00, (70, 180, 80), 0),
    "rain":  WeatherProfile("RAIN", 0.82, 0.72, 0.72, 0.86, 1.55, (48, 105, 95), 45),
    "fog":   WeatherProfile("FOG",  0.70, 0.84, 0.48, 0.78, 1.75, (130, 140, 135), 80),
    "storm": WeatherProfile("STORM",0.62, 0.58, 0.40, 0.62, 2.45, (38, 55, 65), 95),
}


# ------------------------------------------------------------
# Main configuration
# ------------------------------------------------------------
@dataclass
class TrafficConfig:
    game_w: int = 640
    game_h: int = 620
    panel_w: int = 330

    n_lanes: int = 3
    road_width: int = 390

    fps: int = 25
    seconds: int = 20
    out_gif: str = "weather_v2v_traffic_with_images.gif"

    weather_mode: str = "rain"   # clear, rain, fog, storm

    initial_vehicles: int = 30
    max_vehicles: int = 50

    arrival_rate_vpm: Tuple[float, float, float] = (30.0, 58.0, 35.0)

    p_car: float = 0.72
    p_truck: float = 0.20
    p_slow_vehicle: float = 0.08

    p_connected_vehicle: float = 0.90

    car_speed_mean: float = 3.25
    car_speed_std: float = 0.35

    truck_speed_mean: float = 2.65
    truck_speed_std: float = 0.25

    slow_speed_mean: float = 1.55
    slow_speed_std: float = 0.20

    leader_speed_init: float = 3.25
    leader_speed_min: float = 0.80
    leader_speed_max: float = 4.35

    local_sensor_range: float = 120.0
    v2v_lookahead_range: float = 340.0
    safe_front_gap: float = 108.0
    safe_rear_gap: float = 75.0

    max_accel: float = 0.055
    max_decel: float = 0.24
    lane_change_cooldown_frames: int = 18
    lateral_smoothing: float = 0.22

    comm_range: float = 290.0
    channel_base_success: float = 0.96
    channel_decay: float = 260.0
    channel_load_penalty: float = 0.018
    random_packet_loss: float = 0.04

    congestion_lane: int = 1
    congestion_y_min: float = 80.0
    congestion_y_max: float = 360.0
    congestion_speed_factor: float = 0.55

    risk_threshold_lane_change: float = 1.35
    risk_threshold_slowdown: float = 0.88

    enable_random_crashes: bool = True
    base_random_crash_prob: float = 0.0008
    crash_duration_frames: int = 220

    use_uav_supervisor: bool = True
    uav_coverage: float = 285.0
    uav_broadcast_success: float = 0.96

    # --------------------------------------------------------
    # Vehicle image paths
    # --------------------------------------------------------
    car_img_path: str = "car.png"
    taxi_img_path: str = "taxi.png"
    semi_trailer_img_path: str = "semi_trailer.png"
    pickup_truck_img_path: str = "pickup_truck.png"


@dataclass
class Vehicle:
    vid: int
    lane: int
    x: float
    y: float
    speed: float
    desired_speed: float
    length: int
    width: int
    kind: str
    color: Tuple[int, int, int]
    sprite: Optional[pygame.Surface] = None

    connected: bool = True
    is_leader: bool = False
    alive: bool = True
    crashed: bool = False
    crash_timer: int = 0
    lane_cooldown: int = 0
    rx_messages: int = 0

    def rect(self) -> pygame.Rect:
        return pygame.Rect(
            int(self.x - self.width / 2),
            int(self.y - self.length / 2),
            self.width,
            self.length,
        )


@dataclass
class V2VMessage:
    sender_id: int
    source: str
    lane: int
    x: float
    y: float
    speed: float
    gap_ahead: float
    density_ahead: int
    hazard_level: float
    crashed: bool
    weather: str
    timestamp_frame: int


@dataclass
class SimStats:
    spawned: int = 0
    passed: int = 0
    escaped_ahead: int = 0

    collisions: int = 0
    random_crashes: int = 0
    leader_crashes: int = 0

    leader_lane_changes: int = 0
    leader_slow_events: int = 0

    leader_v2v_rx: int = 0
    leader_uav_rx: int = 0

    packets_attempted: int = 0
    packets_delivered: int = 0

    min_leader_gap: float = 1e9

    leader_speed_history: List[float] = field(default_factory=list)
    risk_history: List[Tuple[float, float, float]] = field(default_factory=list)
    crash_events: List[Dict] = field(default_factory=list)


@dataclass
class UAVSupervisor:
    x: float
    y: float
    coverage: float


# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------
def clamp(v: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, v))


def get_weather(cfg: TrafficConfig) -> WeatherProfile:
    return WEATHER_DATABASE.get(cfg.weather_mode.lower(), WEATHER_DATABASE["clear"])


def create_lane_centers(cfg: TrafficConfig) -> List[float]:
    road_x0 = (cfg.game_w - cfg.road_width) // 2
    lane_spacing = cfg.road_width / cfg.n_lanes
    return [road_x0 + lane_spacing * (i + 0.5) for i in range(cfg.n_lanes)]


def distance(a: Vehicle, b: Vehicle) -> float:
    dx = a.x - b.x
    dy = a.y - b.y
    return math.sqrt(dx * dx + dy * dy)


def weather_safe_gap(cfg: TrafficConfig, weather: WeatherProfile) -> float:
    return cfg.safe_front_gap * (1.0 + 1.25 * (1.0 - weather.braking_factor))


def weather_sensor_range(cfg: TrafficConfig, weather: WeatherProfile) -> float:
    return cfg.local_sensor_range * weather.visibility_factor


def nearest_vehicle_ahead(ego: Vehicle, vehicles: List[Vehicle], lane: Optional[int] = None):
    if lane is None:
        lane = ego.lane

    best_vehicle = None
    best_gap = 1e9

    for other in vehicles:
        if other is ego or not other.alive:
            continue
        if other.lane != lane:
            continue

        gap = ego.y - other.y
        if 0.0 < gap < best_gap:
            best_gap = gap
            best_vehicle = other

    return best_vehicle, best_gap


def lane_density_ahead(ego: Vehicle, vehicles: List[Vehicle], lane: int, lookahead: float) -> int:
    count = 0
    for other in vehicles:
        if other is ego or not other.alive:
            continue
        if other.lane != lane:
            continue

        gap = ego.y - other.y
        if 0.0 < gap <= lookahead:
            count += 1

    return count


def lane_is_clear_for_leader(leader: Vehicle, vehicles: List[Vehicle], target_lane: int,
                             cfg: TrafficConfig, weather: WeatherProfile) -> bool:
    front_gap = weather_safe_gap(cfg, weather)
    rear_gap = cfg.safe_rear_gap * (1.0 + 0.8 * (1.0 - weather.visibility_factor))

    for other in vehicles:
        if other is leader or not other.alive:
            continue
        if other.lane != target_lane:
            continue

        dy = leader.y - other.y
        if 0.0 < dy < front_gap:
            return False
        if -rear_gap < dy < 0.0:
            return False

    return True


# ------------------------------------------------------------
# Sprite loading
# ------------------------------------------------------------
def load_sprite(path: str, size: Tuple[int, int], fallback_color=(120, 120, 120), label="CAR") -> pygame.Surface:
    """
    Load and resize a sprite. If the file does not exist, create a placeholder.
    """
    w, h = size

    if os.path.exists(path):
        try:
            img = pygame.image.load(path).convert_alpha()
            img = pygame.transform.smoothscale(img, (w, h))
            return img
        except Exception:
            pass

    # fallback placeholder
    surf = pygame.Surface((w, h), pygame.SRCALPHA)
    surf.fill((*fallback_color, 255))
    pygame.draw.rect(surf, (255, 255, 255), surf.get_rect(), 2, border_radius=8)

    font = pygame.font.Font(None, 18)
    txt = font.render(label, True, (255, 255, 255))
    surf.blit(txt, (4, 4))
    return surf


def build_sprite_bank(cfg: TrafficConfig) -> Dict[str, pygame.Surface]:
    """
    Build sprite bank for all vehicle classes.
    """
    return {
        "leader": load_sprite(cfg.car_img_path, (38, 64), (30, 160, 255), "LEAD"),
        "car": load_sprite(cfg.car_img_path, (34, 58), (200, 200, 200), "CAR"),
        "taxi": load_sprite(cfg.taxi_img_path, (34, 58), (255, 215, 60), "TAXI"),
        "truck": load_sprite(cfg.semi_trailer_img_path, (39, 82), (130, 130, 130), "TRUCK"),
        "pickup": load_sprite(cfg.pickup_truck_img_path, (36, 70), (180, 110, 60), "PICK"),
    }


# ------------------------------------------------------------
# Traffic generation
# ------------------------------------------------------------
def sample_vehicle_type(cfg: TrafficConfig, weather: WeatherProfile):
    u = random.random()

    if u < cfg.p_car:
        # split normal cars between car and taxi appearance
        if random.random() < 0.35:
            kind = "taxi"
            length = 58
            width = 34
            color = (255, 205, 65)
        else:
            kind = "car"
            length = 58
            width = 34
            color = random.choice([(210, 210, 210), (80, 145, 255), (235, 90, 90)])

        speed = random.gauss(cfg.car_speed_mean, cfg.car_speed_std)

    elif u < cfg.p_car + cfg.p_truck:
        kind = "truck"
        length = 82
        width = 39
        speed = random.gauss(cfg.truck_speed_mean, cfg.truck_speed_std)
        color = (120, 120, 125)

    else:
        kind = "pickup"
        length = 70
        width = 36
        speed = random.gauss(cfg.slow_speed_mean, cfg.slow_speed_std)
        color = (190, 105, 45)

    speed *= weather.speed_factor
    speed = clamp(speed, 0.55, cfg.leader_speed_max * weather.speed_factor)

    return kind, length, width, speed, color


def spawn_vehicle(vehicles: List[Vehicle], lane_centers: List[float], cfg: TrafficConfig,
                  weather: WeatherProfile, sprites: Dict[str, pygame.Surface],
                  stats: SimStats, next_id: int, initial=False, forced_lane=None):
    if len(vehicles) >= cfg.max_vehicles:
        return None, next_id

    lane = random.randint(0, cfg.n_lanes - 1) if forced_lane is None else forced_lane

    if initial:
        y = random.uniform(-160.0, cfg.game_h + 130.0)
    else:
        if random.random() < 0.82:
            y = random.uniform(-150.0, 80.0)
        else:
            y = random.uniform(cfg.game_h - 60.0, cfg.game_h + 150.0)

    for other in vehicles:
        if other.lane == lane and abs(other.y - y) < cfg.safe_front_gap:
            return None, next_id

    kind, length, width, desired_speed, color = sample_vehicle_type(cfg, weather)

    connected = random.random() < cfg.p_connected_vehicle

    vehicle = Vehicle(
        vid=next_id,
        lane=lane,
        x=float(lane_centers[lane]),
        y=float(y),
        speed=desired_speed,
        desired_speed=desired_speed,
        length=length,
        width=width,
        kind=kind,
        color=color,
        sprite=sprites[kind],
        connected=connected,
        is_leader=False,
    )

    stats.spawned += 1
    return vehicle, next_id + 1


# ------------------------------------------------------------
# V2V messaging
# ------------------------------------------------------------
def make_vehicle_message(sender: Vehicle, vehicles: List[Vehicle], frame_idx: int,
                         cfg: TrafficConfig, weather: WeatherProfile) -> V2VMessage:
    _, gap_ahead = nearest_vehicle_ahead(sender, vehicles, lane=sender.lane)

    density = lane_density_ahead(sender, vehicles, sender.lane, cfg.v2v_lookahead_range)
    safe_gap = weather_safe_gap(cfg, weather)

    hazard = 0.0

    if sender.crashed:
        hazard += 2.4

    if gap_ahead < safe_gap:
        hazard += 1.20
    elif gap_ahead < 1.8 * safe_gap:
        hazard += 0.55

    hazard += 0.18 * density

    if sender.kind in ["pickup", "truck"]:
        hazard += 0.25

    if weather.name in ["RAIN", "FOG", "STORM"]:
        hazard += 0.30

    hazard = float(clamp(hazard, 0.0, 3.5))

    return V2VMessage(
        sender_id=sender.vid,
        source="V2V",
        lane=sender.lane,
        x=sender.x,
        y=sender.y,
        speed=sender.speed,
        gap_ahead=gap_ahead,
        density_ahead=density,
        hazard_level=hazard,
        crashed=sender.crashed,
        weather=weather.name,
        timestamp_frame=frame_idx,
    )


def simulate_v2v_transfer(vehicles: List[Vehicle], frame_idx: int, cfg: TrafficConfig,
                          weather: WeatherProfile, stats: SimStats):
    received_messages = {v.vid: [] for v in vehicles if v.alive}

    connected = [v for v in vehicles if v.alive and v.connected]
    effective_range = cfg.comm_range * (0.75 + 0.25 * weather.comm_factor)

    for sender in connected:
        msg = make_vehicle_message(sender, vehicles, frame_idx, cfg, weather)

        for receiver in connected:
            if receiver is sender:
                continue

            d = distance(sender, receiver)
            if d > effective_range:
                continue

            channel_load = 0
            for other in connected:
                if other is receiver:
                    continue
                if distance(receiver, other) <= effective_range:
                    channel_load += 1

            success_prob = (
                cfg.channel_base_success *
                weather.comm_factor *
                math.exp(-d / cfg.channel_decay) *
                max(0.12, 1.0 - cfg.channel_load_penalty * channel_load) *
                (1.0 - cfg.random_packet_loss)
            )
            success_prob = clamp(success_prob, 0.02, 0.99)

            stats.packets_attempted += 1

            if random.random() < success_prob:
                received_messages[receiver.vid].append(msg)
                receiver.rx_messages += 1
                stats.packets_delivered += 1

    return received_messages


# ------------------------------------------------------------
# UAV Supervisor
# ------------------------------------------------------------
def update_uav(uav: UAVSupervisor, leader: Vehicle, vehicles: List[Vehicle],
               lane_centers: List[float], cfg: TrafficConfig, weather: WeatherProfile):
    lane_risks = []
    safe_gap = weather_safe_gap(cfg, weather)

    for lane in range(cfg.n_lanes):
        density = lane_density_ahead(leader, vehicles, lane, cfg.v2v_lookahead_range)
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < cfg.v2v_lookahead_range
            for v in vehicles
        )

        risk = 0.20 * density

        if gap < safe_gap:
            risk += 1.35
        elif gap < 1.8 * safe_gap:
            risk += 0.55

        if crashed_ahead:
            risk += 2.0

        lane_risks.append(risk)

    target_lane = int(np.argmax(lane_risks))
    target_x = lane_centers[target_lane]
    target_y = max(70.0, leader.y - 210.0)

    uav.x += 0.06 * (target_x - uav.x)
    uav.y += 0.06 * (target_y - uav.y)


def uav_broadcast_to_leader(uav: UAVSupervisor, leader: Vehicle, vehicles: List[Vehicle],
                            frame_idx: int, cfg: TrafficConfig, weather: WeatherProfile):
    dx = leader.x - uav.x
    dy = leader.y - uav.y
    d = math.sqrt(dx * dx + dy * dy)

    if d > uav.coverage:
        return []

    if random.random() > cfg.uav_broadcast_success * weather.comm_factor:
        return []

    safe_gap = weather_safe_gap(cfg, weather)
    messages = []

    for lane in range(cfg.n_lanes):
        density = lane_density_ahead(leader, vehicles, lane, cfg.v2v_lookahead_range)
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < cfg.v2v_lookahead_range
            for v in vehicles
        )

        hazard = 0.15 * density

        if gap < safe_gap:
            hazard += 1.2
        elif gap < 1.8 * safe_gap:
            hazard += 0.55

        if crashed_ahead:
            hazard += 2.2

        if weather.name in ["RAIN", "FOG", "STORM"]:
            hazard += 0.25

        if hazard > 0.12:
            messages.append(
                V2VMessage(
                    sender_id=-1,
                    source="UAV",
                    lane=lane,
                    x=uav.x,
                    y=max(0.0, leader.y - min(gap, cfg.v2v_lookahead_range)),
                    speed=0.0,
                    gap_ahead=gap,
                    density_ahead=density,
                    hazard_level=float(clamp(hazard, 0.0, 4.0)),
                    crashed=crashed_ahead,
                    weather=weather.name,
                    timestamp_frame=frame_idx,
                )
            )

    return messages


# ------------------------------------------------------------
# Leader control
# ------------------------------------------------------------
def estimate_lane_risk_for_leader(leader: Vehicle, vehicles: List[Vehicle],
                                  messages: List[V2VMessage], cfg: TrafficConfig,
                                  weather: WeatherProfile):
    risks = [0.0 for _ in range(cfg.n_lanes)]

    sensor_range = weather_sensor_range(cfg, weather)
    safe_gap = weather_safe_gap(cfg, weather)

    # Local sensing
    for lane in range(cfg.n_lanes):
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)
        density = lane_density_ahead(leader, vehicles, lane, sensor_range)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < sensor_range
            for v in vehicles
        )

        risks[lane] += 0.22 * density

        if gap < safe_gap:
            risks[lane] += 1.45
        elif gap < 1.8 * safe_gap:
            risks[lane] += 0.62

        if crashed_ahead:
            risks[lane] += 2.5

    # Messages
    for msg in messages:
        if msg.lane < 0 or msg.lane >= cfg.n_lanes:
            continue

        lane = msg.lane
        gap_to_msg = leader.y - msg.y

        if msg.source == "V2V":
            if gap_to_msg < 0.0 or gap_to_msg > cfg.v2v_lookahead_range:
                continue

            distance_weight = math.exp(-gap_to_msg / cfg.v2v_lookahead_range)
            risks[lane] += distance_weight * msg.hazard_level
            risks[lane] += 0.08 * msg.density_ahead

            if msg.crashed:
                risks[lane] += 2.0

            if msg.speed < 0.65 * cfg.leader_speed_init:
                risks[lane] += 0.45

        elif msg.source == "UAV":
            risks[lane] += 0.90 * msg.hazard_level
            risks[lane] += 0.12 * msg.density_ahead
            if msg.crashed:
                risks[lane] += 2.2

    if weather.name == "RAIN":
        risks = [r + 0.12 for r in risks]
    elif weather.name == "FOG":
        risks = [r + 0.25 for r in risks]
    elif weather.name == "STORM":
        risks = [r + 0.42 for r in risks]

    return risks


def control_leader_car(leader: Vehicle, vehicles: List[Vehicle], received_messages: List[V2VMessage],
                       lane_centers: List[float], cfg: TrafficConfig,
                       weather: WeatherProfile, stats: SimStats):
    if leader.lane_cooldown > 0:
        leader.lane_cooldown -= 1

    risks = estimate_lane_risk_for_leader(leader, vehicles, received_messages, cfg, weather)
    stats.risk_history.append(tuple(risks))

    current_risk = risks[leader.lane]
    safest_lane = int(np.argmin(risks))

    changed_lane = False

    if (
        current_risk > cfg.risk_threshold_lane_change and
        safest_lane != leader.lane and
        leader.lane_cooldown == 0 and
        lane_is_clear_for_leader(leader, vehicles, safest_lane, cfg, weather)
    ):
        leader.lane = safest_lane
        leader.lane_cooldown = cfg.lane_change_cooldown_frames
        stats.leader_lane_changes += 1
        changed_lane = True

    _, front_gap = nearest_vehicle_ahead(leader, vehicles, leader.lane)
    stats.min_leader_gap = min(stats.min_leader_gap, front_gap)

    safe_gap = weather_safe_gap(cfg, weather)
    must_slow = (front_gap < safe_gap or current_risk > cfg.risk_threshold_slowdown)

    max_decel = cfg.max_decel * weather.braking_factor
    max_speed = cfg.leader_speed_max * weather.speed_factor

    if must_slow and not changed_lane:
        leader.speed = max(cfg.leader_speed_min, leader.speed - max_decel)
        stats.leader_slow_events += 1
    else:
        leader.speed = min(max_speed, leader.speed + cfg.max_accel)

    target_x = lane_centers[leader.lane]
    leader.x += cfg.lateral_smoothing * (target_x - leader.x)
    stats.leader_speed_history.append(leader.speed)


# ------------------------------------------------------------
# Traffic dynamics and crashes
# ------------------------------------------------------------
def update_background_vehicle_speeds(leader: Vehicle, vehicles: List[Vehicle],
                                     cfg: TrafficConfig, weather: WeatherProfile):
    all_vehicles = [leader] + vehicles
    safe_gap = weather_safe_gap(cfg, weather)

    for v in vehicles:
        if not v.alive:
            continue

        if v.crashed:
            v.speed = 0.0
            v.crash_timer -= 1
            if v.crash_timer <= 0:
                v.alive = False
            continue

        _, gap = nearest_vehicle_ahead(v, all_vehicles, v.lane)

        in_congestion_zone = (
            v.lane == cfg.congestion_lane and
            cfg.congestion_y_min <= v.y <= cfg.congestion_y_max
        )

        desired = v.desired_speed
        if in_congestion_zone:
            desired *= cfg.congestion_speed_factor

        if gap < 0.9 * safe_gap:
            v.speed = max(0.45, v.speed - 0.18 * weather.braking_factor)
        else:
            v.speed = min(desired, v.speed + 0.035)


def update_relative_positions(leader: Vehicle, vehicles: List[Vehicle],
                              lane_centers: List[float], cfg: TrafficConfig):
    for v in vehicles:
        if not v.alive:
            continue

        if v.crashed:
            v.y += leader.speed
        else:
            v.y += leader.speed - v.speed

        target_x = lane_centers[v.lane]
        v.x += 0.10 * (target_x - v.x)


def remove_offscreen_vehicles(vehicles: List[Vehicle], cfg: TrafficConfig, stats: SimStats):
    kept = []

    for v in vehicles:
        if not v.alive:
            continue

        if v.y < -240.0:
            stats.escaped_ahead += 1
            continue

        if v.y > cfg.game_h + 220.0:
            stats.passed += 1
            continue

        kept.append(v)

    return kept


def register_crash(stats: SimStats, frame_idx: int, a: Vehicle, b: Optional[Vehicle],
                   reason: str, cfg: TrafficConfig):
    stats.collisions += 1
    if reason == "random":
        stats.random_crashes += 1

    if a.is_leader or (b is not None and b.is_leader):
        stats.leader_crashes += 1

    if not a.is_leader:
        a.crashed = True
        a.speed = 0.0
        a.crash_timer = cfg.crash_duration_frames

    if b is not None and not b.is_leader:
        b.crashed = True
        b.speed = 0.0
        b.crash_timer = cfg.crash_duration_frames

    stats.crash_events.append({
        "frame": int(frame_idx),
        "reason": reason,
        "vehicle_a": int(a.vid),
        "vehicle_b": int(b.vid) if b is not None else None,
        "lane": int(a.lane),
        "x": float(a.x),
        "y": float(a.y),
    })


def detect_collisions_and_random_crashes(leader: Vehicle, vehicles: List[Vehicle],
                                         cfg: TrafficConfig, weather: WeatherProfile,
                                         stats: SimStats, frame_idx: int):
    all_vehicles = [leader] + vehicles

    # Physical collisions
    for i in range(len(all_vehicles)):
        a = all_vehicles[i]
        if not a.alive:
            continue

        ra = a.rect()

        for j in range(i + 1, len(all_vehicles)):
            b = all_vehicles[j]
            if not b.alive:
                continue
            if a.crashed and b.crashed:
                continue

            if ra.colliderect(b.rect()):
                register_crash(stats, frame_idx, a, b, "collision", cfg)
                if leader in [a, b]:
                    leader.speed = max(cfg.leader_speed_min, leader.speed * 0.45)

    # Random crashes
    if not cfg.enable_random_crashes:
        return

    for v in vehicles:
        if not v.alive or v.crashed:
            continue

        _, gap = nearest_vehicle_ahead(v, all_vehicles, v.lane)
        local_density = lane_density_ahead(v, all_vehicles, v.lane, cfg.local_sensor_range)

        in_congestion_zone = (
            v.lane == cfg.congestion_lane and
            cfg.congestion_y_min <= v.y <= cfg.congestion_y_max
        )

        risk = cfg.base_random_crash_prob
        risk *= weather.crash_factor
        risk *= 1.0 + 0.25 * local_density

        if gap < weather_safe_gap(cfg, weather):
            risk *= 3.0
        if in_congestion_zone:
            risk *= 2.0

        if random.random() < risk:
            register_crash(stats, frame_idx, v, None, "random", cfg)


# ------------------------------------------------------------
# Drawing
# ------------------------------------------------------------
def draw_vehicle(surface: pygame.Surface, v: Vehicle, is_leader=False):
    rect = v.rect()

    if v.sprite is not None:
        img = pygame.transform.smoothscale(v.sprite, (rect.width, rect.height))
        surface.blit(img, rect.topleft)
    else:
        pygame.draw.rect(surface, v.color, rect, border_radius=7)

    if v.crashed:
        pygame.draw.rect(surface, (255, 50, 50), rect, 3, border_radius=7)
        pygame.draw.line(surface, (255, 0, 0), rect.topleft, rect.bottomright, 3)
        pygame.draw.line(surface, (255, 0, 0), rect.topright, rect.bottomleft, 3)
        font = pygame.font.Font(None, 18)
        txt = font.render("CRASH", True, (255, 255, 255))
        surface.blit(txt, (rect.x - 2, rect.y - 18))
        return

    if is_leader:
        pygame.draw.rect(surface, (255, 255, 255), rect, 2, border_radius=7)
        font = pygame.font.Font(None, 18)
        txt = font.render("LEADER", True, (255, 255, 255))
        surface.blit(txt, (rect.x - 8, rect.y - 18))
    else:
        dot = (40, 255, 90) if v.connected else (255, 70, 70)
        pygame.draw.circle(surface, dot, (int(v.x), int(v.y - v.length / 2 - 7)), 4)


def draw_uav(surface: pygame.Surface, uav: UAVSupervisor, cfg: TrafficConfig):
    overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)

    pygame.draw.circle(
        overlay,
        (80, 190, 255, 32),
        (int(uav.x), int(uav.y)),
        int(uav.coverage),
    )
    surface.blit(overlay, (0, 0))

    pygame.draw.circle(surface, (60, 190, 255), (int(uav.x), int(uav.y)), 11)
    pygame.draw.circle(surface, (255, 255, 255), (int(uav.x), int(uav.y)), 4)

    pygame.draw.line(surface, (255, 255, 255), (int(uav.x - 17), int(uav.y)),
                     (int(uav.x + 17), int(uav.y)), 2)
    pygame.draw.line(surface, (255, 255, 255), (int(uav.x), int(uav.y - 17)),
                     (int(uav.x), int(uav.y + 17)), 2)


def draw_weather_overlay(screen: pygame.Surface, cfg: TrafficConfig,
                         weather: WeatherProfile, frame_idx: int):
    if weather.overlay_alpha <= 0:
        return

    overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)

    if weather.name == "RAIN":
        overlay.fill((40, 70, 90, weather.overlay_alpha))
        for _ in range(45):
            x = random.randint(0, cfg.game_w)
            y = random.randint(0, cfg.game_h)
            pygame.draw.line(overlay, (180, 210, 255, 120), (x, y), (x - 5, y + 14), 1)

    elif weather.name == "FOG":
        overlay.fill((210, 210, 210, weather.overlay_alpha))

    elif weather.name == "STORM":
        overlay.fill((30, 35, 50, weather.overlay_alpha))
        for _ in range(65):
            x = random.randint(0, cfg.game_w)
            y = random.randint(0, cfg.game_h)
            pygame.draw.line(overlay, (160, 190, 255, 135), (x, y), (x - 7, y + 17), 1)
        if frame_idx % 70 < 4:
            overlay.fill((255, 255, 255, 70))

    screen.blit(overlay, (0, 0))


def make_risk_panel(risks: Tuple[float, float, float], cfg: TrafficConfig, panel_w: int, panel_h: int):
    surf = pygame.Surface((panel_w, panel_h))
    surf.fill((18, 18, 28))

    font = pygame.font.Font(None, 21)
    small = pygame.font.Font(None, 18)

    surf.blit(font.render("Leader lane-risk estimate", True, (255, 255, 255)), (10, 10))
    max_r = max(1.0, max(risks))

    for i, risk in enumerate(risks):
        y = 48 + i * 52
        w = int((panel_w - 90) * risk / max_r)

        if risk > cfg.risk_threshold_lane_change:
            color = (255, 80, 80)
        elif risk > cfg.risk_threshold_slowdown:
            color = (255, 190, 70)
        else:
            color = (80, 220, 110)

        surf.blit(small.render(f"Lane {i}", True, (230, 230, 230)), (10, y + 5))
        pygame.draw.rect(surf, (60, 60, 75), (72, y, panel_w - 90, 24))
        pygame.draw.rect(surf, color, (72, y, w, 24))
        surf.blit(small.render(f"{risk:.2f}", True, (255, 255, 255)), (panel_w - 48, y + 4))

    return surf


# ------------------------------------------------------------
# Main simulation
# ------------------------------------------------------------
def run_weather_v2v_traffic_sim(cfg: Optional[TrafficConfig] = None, seed: int = 12):
    if cfg is None:
        cfg = TrafficConfig()

    random.seed(seed)
    np.random.seed(seed)

    weather = get_weather(cfg)

    pygame.display.init()
    pygame.font.init()

    # small hidden screen requirement for image loading in some environments
    screen = pygame.display.set_mode((cfg.game_w + cfg.panel_w, cfg.game_h))
    clock = pygame.time.Clock()

    font = pygame.font.Font(None, 22)
    small = pygame.font.Font(None, 18)

    sprites = build_sprite_bank(cfg)
    lane_centers = create_lane_centers(cfg)

    road_x0 = int((cfg.game_w - cfg.road_width) / 2)
    road_rect = pygame.Rect(road_x0, 0, cfg.road_width, cfg.game_h)

    stats = SimStats()
    leader_y = cfg.game_h - 115.0

    leader = Vehicle(
        vid=0,
        lane=1,
        x=float(lane_centers[1]),
        y=leader_y,
        speed=cfg.leader_speed_init * weather.speed_factor,
        desired_speed=cfg.leader_speed_max * weather.speed_factor,
        length=64,
        width=38,
        kind="leader",
        color=(35, 165, 255),
        sprite=sprites["leader"],
        connected=True,
        is_leader=True,
    )

    vehicles: List[Vehicle] = []
    next_id = 1

    for _ in range(cfg.initial_vehicles):
        v, next_id = spawn_vehicle(vehicles, lane_centers, cfg, weather, sprites, stats, next_id, initial=True)
        if v is not None:
            vehicles.append(v)

    uav = UAVSupervisor(
        x=float(lane_centers[1]),
        y=leader.y - 220.0,
        coverage=cfg.uav_coverage,
    )

    lane_marker_offset = 0.0
    n_frames = int(cfg.fps * cfg.seconds)

    with imageio.get_writer(cfg.out_gif, mode="I", fps=cfg.fps) as writer:
        for frame_idx in range(n_frames):
            clock.tick(cfg.fps)
            pygame.event.pump()

            # --------------------------------------------
            # Statistical arrivals
            # --------------------------------------------
            for lane in range(cfg.n_lanes):
                weather_arrival_factor = 0.88 if weather.name in ["RAIN", "FOG", "STORM"] else 1.0
                arrival_per_frame = cfg.arrival_rate_vpm[lane] * weather_arrival_factor / 60.0 / cfg.fps

                if random.random() < arrival_per_frame:
                    v, next_id = spawn_vehicle(
                        vehicles, lane_centers, cfg, weather, sprites, stats, next_id,
                        initial=False, forced_lane=lane
                    )
                    if v is not None:
                        vehicles.append(v)

            # --------------------------------------------
            # V2V
            # --------------------------------------------
            all_vehicles = [leader] + vehicles

            received = simulate_v2v_transfer(all_vehicles, frame_idx, cfg, weather, stats)
            leader_messages = received.get(leader.vid, [])

            stats.leader_v2v_rx += len([m for m in leader_messages if m.source == "V2V"])

            # --------------------------------------------
            # UAV
            # --------------------------------------------
            if cfg.use_uav_supervisor:
                update_uav(uav, leader, vehicles, lane_centers, cfg, weather)
                uav_msgs = uav_broadcast_to_leader(uav, leader, vehicles, frame_idx, cfg, weather)
                leader_messages.extend(uav_msgs)
                stats.leader_uav_rx += len(uav_msgs)

            # --------------------------------------------
            # Leader control
            # --------------------------------------------
            control_leader_car(leader, vehicles, leader_messages, lane_centers, cfg, weather, stats)

            # --------------------------------------------
            # Traffic dynamics
            # --------------------------------------------
            update_background_vehicle_speeds(leader, vehicles, cfg, weather)
            update_relative_positions(leader, vehicles, lane_centers, cfg)
            leader.y = leader_y

            detect_collisions_and_random_crashes(leader, vehicles, cfg, weather, stats, frame_idx)
            vehicles = remove_offscreen_vehicles(vehicles, cfg, stats)

            # --------------------------------------------
            # Rendering
            # --------------------------------------------
            screen.fill(weather.background_color)
            pygame.draw.rect(screen, (82, 82, 86), road_rect)

            # Congestion overlay
            congestion_rect = pygame.Rect(
                int(road_x0 + cfg.congestion_lane * cfg.road_width / cfg.n_lanes),
                int(cfg.congestion_y_min),
                int(cfg.road_width / cfg.n_lanes),
                int(cfg.congestion_y_max - cfg.congestion_y_min),
            )

            congestion_overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)
            pygame.draw.rect(congestion_overlay, (255, 120, 30, 55), congestion_rect)
            screen.blit(congestion_overlay, (0, 0))

            # Road borders
            pygame.draw.rect(screen, (255, 220, 40), (road_x0 - 8, 0, 8, cfg.game_h))
            pygame.draw.rect(screen, (255, 220, 40), (road_x0 + cfg.road_width, 0, 8, cfg.game_h))

            # Lane markers
            lane_marker_offset += max(1.0, leader.speed * 2.0)
            if lane_marker_offset > 96:
                lane_marker_offset = 0.0

            for y in range(-120, cfg.game_h + 120, 96):
                yy = y + int(lane_marker_offset)
                for boundary in range(1, cfg.n_lanes):
                    x = int(road_x0 + boundary * cfg.road_width / cfg.n_lanes)
                    pygame.draw.rect(screen, (245, 245, 245), (x - 4, yy, 8, 48))

            # UAV
            if cfg.use_uav_supervisor:
                draw_uav(screen, uav, cfg)

            # Links
            for msg in leader_messages:
                if msg.source == "V2V":
                    color = (80, 255, 140)
                    width = 1
                    target = (int(msg.x), int(msg.y))
                else:
                    color = (80, 190, 255)
                    width = 2
                    target = (int(uav.x), int(uav.y))

                pygame.draw.line(screen, color, (int(leader.x), int(leader.y)), target, width)

            # Vehicles
            for v in vehicles:
                draw_vehicle(screen, v, is_leader=False)
            draw_vehicle(screen, leader, is_leader=True)

            draw_weather_overlay(screen, cfg, weather, frame_idx)

            # --------------------------------------------
            # HUD
            # --------------------------------------------
            current_risks = stats.risk_history[-1] if stats.risk_history else (0.0, 0.0, 0.0)
            connected_count = sum(1 for v in vehicles if v.connected)
            crashed_count = sum(1 for v in vehicles if v.crashed)

            packet_rate = stats.packets_delivered / stats.packets_attempted if stats.packets_attempted > 0 else 0.0

            screen.blit(font.render("Weather-Aware V2V Traffic and Crash Avoidance", True, (255, 255, 255)), (12, 10))
            screen.blit(font.render(f"Weather={weather.name} | Leader lane={leader.lane} | speed={leader.speed:.2f}", True, (255, 255, 255)), (12, 34))
            screen.blit(font.render(f"Vehicles={len(vehicles)} | connected={connected_count} | crashed={crashed_count}", True, (255, 255, 255)), (12, 58))
            screen.blit(font.render(f"V2V RX={stats.leader_v2v_rx} | UAV RX={stats.leader_uav_rx} | PDR={100.0 * packet_rate:.1f}%", True, (255, 255, 255)), (12, 82))
            screen.blit(font.render(f"Lane changes={stats.leader_lane_changes} | slow events={stats.leader_slow_events}", True, (255, 255, 255)), (12, 106))
            screen.blit(font.render(f"Crashes={stats.collisions} | random={stats.random_crashes} | leader={stats.leader_crashes}", True, (255, 255, 255)), (12, 130))

            legend_y = cfg.game_h - 92
            pygame.draw.circle(screen, (40, 255, 90), (20, legend_y), 5)
            screen.blit(small.render("Connected V2V vehicle", True, (255, 255, 255)), (34, legend_y - 7))

            pygame.draw.circle(screen, (255, 70, 70), (20, legend_y + 22), 5)
            screen.blit(small.render("Non-connected vehicle", True, (255, 255, 255)), (34, legend_y + 15))

            pygame.draw.rect(screen, (255, 50, 50), (15, legend_y + 40, 12, 12), 2)
            screen.blit(small.render("Crash / blocked vehicle", True, (255, 255, 255)), (34, legend_y + 37))
            screen.blit(small.render("Orange area: congestion zone", True, (255, 255, 255)), (34, legend_y + 59))

            # Side panel
            panel_x = cfg.game_w
            pygame.draw.rect(screen, (18, 18, 28), (panel_x, 0, cfg.panel_w, cfg.game_h))

            risk_panel = make_risk_panel(current_risks, cfg, cfg.panel_w, 230)
            screen.blit(risk_panel, (panel_x, 0))

            y0 = 250
            screen.blit(font.render("Simulation statistics", True, (255, 255, 255)), (panel_x + 10, y0))

            side_lines = [
                f"Weather mode     : {weather.name}",
                f"Spawned vehicles : {stats.spawned}",
                f"Passed vehicles  : {stats.passed}",
                f"Escaped ahead    : {stats.escaped_ahead}",
                f"Active vehicles  : {len(vehicles)}",
                f"Crash obstacles  : {crashed_count}",
                f"Packets attempted: {stats.packets_attempted}",
                f"Packets delivered: {stats.packets_delivered}",
                f"Leader V2V RX    : {stats.leader_v2v_rx}",
                f"Leader UAV RX    : {stats.leader_uav_rx}",
                f"Min leader gap   : {stats.min_leader_gap:.1f}px",
            ]

            for i, line in enumerate(side_lines):
                screen.blit(small.render(line, True, (235, 235, 235)), (panel_x + 10, y0 + 34 + 22 * i))

            box_y = cfg.game_h - 150
            pygame.draw.rect(screen, (30, 30, 45), (panel_x + 10, box_y, cfg.panel_w - 20, 134), border_radius=8)

            info = [
                "Leader control:",
                "1) Receive V2V/UAV traffic alerts",
                "2) Estimate weather-aware lane risk",
                "3) Avoid congestion and crash zones",
                "4) Slow down when lane change is unsafe",
                "5) Use vehicle image sprites",
            ]

            for i, line in enumerate(info):
                screen.blit(small.render(line, True, (255, 255, 255)), (panel_x + 22, box_y + 12 + 22 * i))

            pygame.display.flip()

            arr = pygame.surfarray.array3d(screen)
            arr = np.transpose(arr, (1, 0, 2))
            writer.append_data(arr)

    pygame.quit()

    print(f"[DONE] GIF saved: {cfg.out_gif}")
    print(f"[WEATHER] {weather.name}")
    print(f"[STATS] Spawned vehicles     : {stats.spawned}")
    print(f"[STATS] Passed vehicles      : {stats.passed}")
    print(f"[STATS] Total crashes        : {stats.collisions}")
    print(f"[STATS] Random crashes       : {stats.random_crashes}")
    print(f"[STATS] Leader crashes       : {stats.leader_crashes}")
    print(f"[STATS] Leader lane changes  : {stats.leader_lane_changes}")
    print(f"[STATS] Leader slow events   : {stats.leader_slow_events}")
    print(f"[STATS] Packets attempted    : {stats.packets_attempted}")
    print(f"[STATS] Packets delivered    : {stats.packets_delivered}")

    if stats.packets_attempted > 0:
        pdr = 100.0 * stats.packets_delivered / stats.packets_attempted
        print(f"[STATS] Packet delivery rate : {pdr:.2f}%")

    if stats.min_leader_gap < 1e9:
        print(f"[STATS] Minimum leader gap   : {stats.min_leader_gap:.2f}px")

    try:
        from IPython.display import Image, display
        display(Image(filename=cfg.out_gif))
    except Exception:
        pass

    return cfg.out_gif, stats


# ------------------------------------------------------------
# Demo
# ------------------------------------------------------------
def run_demo():
    cfg = TrafficConfig(
        out_gif="weather_v2v_traffic_with_images.gif",
        weather_mode="storm",   # try clear, rain, fog, storm

        car_img_path="/home/cbeario/DeepPrior/car.png",
        taxi_img_path="/home/cbeario/DeepPrior/taxi.png",
        semi_trailer_img_path="/home/cbeario/DeepPrior/semi_trailer.png",
        pickup_truck_img_path="/home/cbeario/DeepPrior/pickup_truck.png",
    )

    return run_weather_v2v_traffic_sim(cfg=cfg, seed=12)


if __name__ == "__main__":
    run_demo()

#**Agent-driver**

In [ ]:
#!/usr/bin/env python3
"""
Weather + Congestion + Crash + V2V/UAV + DQN Leader-Car Simulation
with real vehicle image sprites.

Required image files, if available:
    car.png
    taxi.png
    semi_trailer.png
    pickup_truck.png

The simulator still runs without these files by using fallback colored sprites.

Main features:
    - Statistical multi-lane traffic generation.
    - Weather modes: clear, rain, fog, storm.
    - Weather affects speed, braking, visibility, V2V reliability, and crash risk.
    - V2V packet transfer among connected vehicles.
    - UAV/drone supervisor broadcasts lane-level hazard alerts.
    - DQN leader agent chooses high-level driving actions.
    - Safety shield prevents unsafe DQN lane changes.
    - Crash and congestion zones are simulated.
    - Animated GIF export.
    - Google Colab control panel with Start, Stop, Reset, and live scope indication.

Dependencies:
    pip install numpy pygame imageio torch ipywidgets

Run:
    python weather_v2v_traffic_dqn_v2v_uav.py
"""

import os
import math
import random
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pygame
import imageio.v2 as imageio

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    torch = None
    nn = None
    optim = None


# ------------------------------------------------------------
# Headless support for Linux / Google Colab
# ------------------------------------------------------------
if os.name != "nt" and os.environ.get("DISPLAY", "") == "":
    os.environ.setdefault("SDL_VIDEODRIVER", "dummy")


# ------------------------------------------------------------
# Weather database
# ------------------------------------------------------------
@dataclass
class WeatherProfile:
    name: str
    speed_factor: float
    braking_factor: float
    visibility_factor: float
    comm_factor: float
    crash_factor: float
    background_color: Tuple[int, int, int]
    overlay_alpha: int


WEATHER_DATABASE: Dict[str, WeatherProfile] = {
    "clear": WeatherProfile("CLEAR", 1.00, 1.00, 1.00, 1.00, 1.00, (70, 180, 80), 0),
    "rain": WeatherProfile("RAIN", 0.82, 0.72, 0.72, 0.86, 1.55, (48, 105, 95), 45),
    "fog": WeatherProfile("FOG", 0.70, 0.84, 0.48, 0.78, 1.75, (130, 140, 135), 80),
    "storm": WeatherProfile("STORM", 0.62, 0.58, 0.40, 0.62, 2.45, (38, 55, 65), 95),
}


# ------------------------------------------------------------
# Main configuration
# ------------------------------------------------------------
@dataclass
class TrafficConfig:
    game_w: int = 640
    game_h: int = 620
    panel_w: int = 360

    n_lanes: int = 3
    road_width: int = 390

    fps: int = 25
    seconds: int = 20
    out_gif: str = "weather_v2v_traffic_dqn_v2v_uav.gif"

    weather_mode: str = "rain"  # clear, rain, fog, storm

    initial_vehicles: int = 34
    max_vehicles: int = 54

    arrival_rate_vpm: Tuple[float, float, float] = (32.0, 62.0, 36.0)

    p_car: float = 0.72
    p_truck: float = 0.20
    p_slow_vehicle: float = 0.08

    p_connected_vehicle: float = 0.92

    car_speed_mean: float = 3.25
    car_speed_std: float = 0.35

    truck_speed_mean: float = 2.65
    truck_speed_std: float = 0.25

    slow_speed_mean: float = 1.55
    slow_speed_std: float = 0.20

    leader_speed_init: float = 3.25
    leader_speed_min: float = 0.80
    leader_speed_max: float = 4.35

    local_sensor_range: float = 120.0
    v2v_lookahead_range: float = 340.0
    safe_front_gap: float = 108.0
    safe_rear_gap: float = 75.0

    max_accel: float = 0.055
    max_decel: float = 0.24
    lane_change_cooldown_frames: int = 18
    lateral_smoothing: float = 0.22

    comm_range: float = 290.0
    channel_base_success: float = 0.96
    channel_decay: float = 260.0
    channel_load_penalty: float = 0.018
    random_packet_loss: float = 0.04

    congestion_lane: int = 1
    congestion_y_min: float = 80.0
    congestion_y_max: float = 360.0
    congestion_speed_factor: float = 0.52

    risk_threshold_lane_change: float = 1.35
    risk_threshold_slowdown: float = 0.88

    enable_random_crashes: bool = True
    base_random_crash_prob: float = 0.0008
    crash_duration_frames: int = 220

    use_uav_supervisor: bool = True
    uav_coverage: float = 290.0
    uav_broadcast_success: float = 0.96

    # --------------------------------------------------------
    # DQN leader-agent control
    # --------------------------------------------------------
    use_dqn_agent: bool = True
    dqn_training: bool = True

    dqn_gamma: float = 0.97
    dqn_lr: float = 1e-3
    dqn_batch_size: int = 64
    dqn_buffer_size: int = 20000

    dqn_epsilon_start: float = 0.70
    dqn_epsilon_min: float = 0.05
    dqn_epsilon_decay: float = 0.992

    dqn_target_update_frames: int = 100
    dqn_warmup_steps: int = 180

    dqn_load_path: str = ""
    dqn_save_path: str = "leader_dqn_v2v_uav.pt"

    # --------------------------------------------------------
    # Vehicle image paths
    # --------------------------------------------------------
    car_img_path: str = "car.png"
    taxi_img_path: str = "taxi.png"
    semi_trailer_img_path: str = "semi_trailer.png"
    pickup_truck_img_path: str = "pickup_truck.png"


@dataclass
class Vehicle:
    vid: int
    lane: int
    x: float
    y: float
    speed: float
    desired_speed: float
    length: int
    width: int
    kind: str
    color: Tuple[int, int, int]
    sprite: Optional[pygame.Surface] = None

    connected: bool = True
    is_leader: bool = False
    alive: bool = True
    crashed: bool = False
    crash_timer: int = 0
    lane_cooldown: int = 0
    rx_messages: int = 0

    def rect(self) -> pygame.Rect:
        return pygame.Rect(
            int(self.x - self.width / 2),
            int(self.y - self.length / 2),
            self.width,
            self.length,
        )


@dataclass
class V2VMessage:
    sender_id: int
    source: str
    lane: int
    x: float
    y: float
    speed: float
    gap_ahead: float
    density_ahead: int
    hazard_level: float
    crashed: bool
    weather: str
    timestamp_frame: int


@dataclass
class SimStats:
    spawned: int = 0
    passed: int = 0
    escaped_ahead: int = 0

    collisions: int = 0
    random_crashes: int = 0
    leader_crashes: int = 0

    leader_lane_changes: int = 0
    leader_slow_events: int = 0

    leader_v2v_rx: int = 0
    leader_uav_rx: int = 0

    packets_attempted: int = 0
    packets_delivered: int = 0

    min_leader_gap: float = 1e9

    leader_speed_history: List[float] = field(default_factory=list)
    risk_history: List[Tuple[float, float, float]] = field(default_factory=list)
    crash_events: List[Dict] = field(default_factory=list)

    dqn_loss_history: List[float] = field(default_factory=list)
    dqn_reward_history: List[float] = field(default_factory=list)
    dqn_action_history: List[str] = field(default_factory=list)


@dataclass
class UAVSupervisor:
    x: float
    y: float
    coverage: float


# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------
def clamp(v: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, v))


def get_weather(cfg: TrafficConfig) -> WeatherProfile:
    return WEATHER_DATABASE.get(cfg.weather_mode.lower(), WEATHER_DATABASE["clear"])


def create_lane_centers(cfg: TrafficConfig) -> List[float]:
    road_x0 = (cfg.game_w - cfg.road_width) // 2
    lane_spacing = cfg.road_width / cfg.n_lanes
    return [road_x0 + lane_spacing * (i + 0.5) for i in range(cfg.n_lanes)]


def distance(a: Vehicle, b: Vehicle) -> float:
    dx = a.x - b.x
    dy = a.y - b.y
    return math.sqrt(dx * dx + dy * dy)


def weather_safe_gap(cfg: TrafficConfig, weather: WeatherProfile) -> float:
    return cfg.safe_front_gap * (1.0 + 1.25 * (1.0 - weather.braking_factor))


def weather_sensor_range(cfg: TrafficConfig, weather: WeatherProfile) -> float:
    return cfg.local_sensor_range * weather.visibility_factor


def nearest_vehicle_ahead(ego: Vehicle, vehicles: List[Vehicle], lane: Optional[int] = None):
    if lane is None:
        lane = ego.lane

    best_vehicle = None
    best_gap = 1e9

    for other in vehicles:
        if other is ego or not other.alive:
            continue
        if other.lane != lane:
            continue

        gap = ego.y - other.y
        if 0.0 < gap < best_gap:
            best_gap = gap
            best_vehicle = other

    return best_vehicle, best_gap


def lane_density_ahead(ego: Vehicle, vehicles: List[Vehicle], lane: int, lookahead: float) -> int:
    count = 0
    for other in vehicles:
        if other is ego or not other.alive:
            continue
        if other.lane != lane:
            continue

        gap = ego.y - other.y
        if 0.0 < gap <= lookahead:
            count += 1

    return count


def lane_is_clear_for_leader(
    leader: Vehicle,
    vehicles: List[Vehicle],
    target_lane: int,
    cfg: TrafficConfig,
    weather: WeatherProfile,
) -> bool:
    front_gap = weather_safe_gap(cfg, weather)
    rear_gap = cfg.safe_rear_gap * (1.0 + 0.8 * (1.0 - weather.visibility_factor))

    for other in vehicles:
        if other is leader or not other.alive or other.crashed:
            continue
        if other.lane != target_lane:
            continue

        dy = leader.y - other.y
        if 0.0 < dy < front_gap:
            return False
        if -rear_gap < dy < 0.0:
            return False

    return True


# ------------------------------------------------------------
# Sprite loading
# ------------------------------------------------------------
def load_sprite(path: str, size: Tuple[int, int], fallback_color=(120, 120, 120), label="CAR") -> pygame.Surface:
    """Load and resize a sprite. If the file does not exist, create a placeholder."""
    w, h = size

    if os.path.exists(path):
        try:
            img = pygame.image.load(path).convert_alpha()
            img = pygame.transform.smoothscale(img, (w, h))
            return img
        except Exception:
            pass

    surf = pygame.Surface((w, h), pygame.SRCALPHA)
    surf.fill((*fallback_color, 255))
    pygame.draw.rect(surf, (255, 255, 255), surf.get_rect(), 2, border_radius=8)

    font = pygame.font.Font(None, 18)
    txt = font.render(label, True, (255, 255, 255))
    surf.blit(txt, (4, 4))
    return surf


def build_sprite_bank(cfg: TrafficConfig) -> Dict[str, pygame.Surface]:
    """Build sprite bank for all vehicle classes."""
    return {
        "leader": load_sprite(cfg.car_img_path, (38, 64), (30, 160, 255), "LEAD"),
        "car": load_sprite(cfg.car_img_path, (34, 58), (200, 200, 200), "CAR"),
        "taxi": load_sprite(cfg.taxi_img_path, (34, 58), (255, 215, 60), "TAXI"),
        "truck": load_sprite(cfg.semi_trailer_img_path, (39, 82), (130, 130, 130), "TRUCK"),
        "pickup": load_sprite(cfg.pickup_truck_img_path, (36, 70), (180, 110, 60), "PICK"),
    }


# ------------------------------------------------------------
# Traffic generation
# ------------------------------------------------------------
def sample_vehicle_type(cfg: TrafficConfig, weather: WeatherProfile):
    u = random.random()

    if u < cfg.p_car:
        if random.random() < 0.35:
            kind = "taxi"
            length = 58
            width = 34
            color = (255, 205, 65)
        else:
            kind = "car"
            length = 58
            width = 34
            color = random.choice([(210, 210, 210), (80, 145, 255), (235, 90, 90)])

        speed = random.gauss(cfg.car_speed_mean, cfg.car_speed_std)

    elif u < cfg.p_car + cfg.p_truck:
        kind = "truck"
        length = 82
        width = 39
        speed = random.gauss(cfg.truck_speed_mean, cfg.truck_speed_std)
        color = (120, 120, 125)

    else:
        kind = "pickup"
        length = 70
        width = 36
        speed = random.gauss(cfg.slow_speed_mean, cfg.slow_speed_std)
        color = (190, 105, 45)

    speed *= weather.speed_factor
    speed = clamp(speed, 0.55, cfg.leader_speed_max * weather.speed_factor)

    return kind, length, width, speed, color


def spawn_vehicle(
    vehicles: List[Vehicle],
    lane_centers: List[float],
    cfg: TrafficConfig,
    weather: WeatherProfile,
    sprites: Dict[str, pygame.Surface],
    stats: SimStats,
    next_id: int,
    initial=False,
    forced_lane=None,
):
    if len(vehicles) >= cfg.max_vehicles:
        return None, next_id

    lane = random.randint(0, cfg.n_lanes - 1) if forced_lane is None else forced_lane

    if initial:
        y = random.uniform(-160.0, cfg.game_h + 130.0)
    else:
        if random.random() < 0.82:
            y = random.uniform(-150.0, 80.0)
        else:
            y = random.uniform(cfg.game_h - 60.0, cfg.game_h + 150.0)

    for other in vehicles:
        if other.lane == lane and abs(other.y - y) < cfg.safe_front_gap:
            return None, next_id

    kind, length, width, desired_speed, color = sample_vehicle_type(cfg, weather)
    connected = random.random() < cfg.p_connected_vehicle

    vehicle = Vehicle(
        vid=next_id,
        lane=lane,
        x=float(lane_centers[lane]),
        y=float(y),
        speed=desired_speed,
        desired_speed=desired_speed,
        length=length,
        width=width,
        kind=kind,
        color=color,
        sprite=sprites[kind],
        connected=connected,
        is_leader=False,
    )

    stats.spawned += 1
    return vehicle, next_id + 1


# ------------------------------------------------------------
# V2V messaging
# ------------------------------------------------------------
def make_vehicle_message(sender: Vehicle, vehicles: List[Vehicle], frame_idx: int, cfg: TrafficConfig, weather: WeatherProfile) -> V2VMessage:
    _, gap_ahead = nearest_vehicle_ahead(sender, vehicles, lane=sender.lane)

    density = lane_density_ahead(sender, vehicles, sender.lane, cfg.v2v_lookahead_range)
    safe_gap = weather_safe_gap(cfg, weather)

    hazard = 0.0

    if sender.crashed:
        hazard += 2.4

    if gap_ahead < safe_gap:
        hazard += 1.20
    elif gap_ahead < 1.8 * safe_gap:
        hazard += 0.55

    hazard += 0.18 * density

    if sender.kind in ["pickup", "truck"]:
        hazard += 0.25

    if weather.name in ["RAIN", "FOG", "STORM"]:
        hazard += 0.30

    hazard = float(clamp(hazard, 0.0, 3.5))

    return V2VMessage(
        sender_id=sender.vid,
        source="V2V",
        lane=sender.lane,
        x=sender.x,
        y=sender.y,
        speed=sender.speed,
        gap_ahead=gap_ahead,
        density_ahead=density,
        hazard_level=hazard,
        crashed=sender.crashed,
        weather=weather.name,
        timestamp_frame=frame_idx,
    )


def simulate_v2v_transfer(vehicles: List[Vehicle], frame_idx: int, cfg: TrafficConfig, weather: WeatherProfile, stats: SimStats):
    received_messages = {v.vid: [] for v in vehicles if v.alive}

    connected = [v for v in vehicles if v.alive and v.connected]
    effective_range = cfg.comm_range * (0.75 + 0.25 * weather.comm_factor)

    for sender in connected:
        msg = make_vehicle_message(sender, vehicles, frame_idx, cfg, weather)

        for receiver in connected:
            if receiver is sender:
                continue

            d = distance(sender, receiver)
            if d > effective_range:
                continue

            channel_load = 0
            for other in connected:
                if other is receiver:
                    continue
                if distance(receiver, other) <= effective_range:
                    channel_load += 1

            success_prob = (
                cfg.channel_base_success
                * weather.comm_factor
                * math.exp(-d / cfg.channel_decay)
                * max(0.12, 1.0 - cfg.channel_load_penalty * channel_load)
                * (1.0 - cfg.random_packet_loss)
            )
            success_prob = clamp(success_prob, 0.02, 0.99)

            stats.packets_attempted += 1

            if random.random() < success_prob:
                received_messages[receiver.vid].append(msg)
                receiver.rx_messages += 1
                stats.packets_delivered += 1

    return received_messages


# ------------------------------------------------------------
# UAV Supervisor
# ------------------------------------------------------------
def update_uav(uav: UAVSupervisor, leader: Vehicle, vehicles: List[Vehicle], lane_centers: List[float], cfg: TrafficConfig, weather: WeatherProfile):
    lane_risks = []
    safe_gap = weather_safe_gap(cfg, weather)

    for lane in range(cfg.n_lanes):
        density = lane_density_ahead(leader, vehicles, lane, cfg.v2v_lookahead_range)
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < cfg.v2v_lookahead_range
            for v in vehicles
        )

        risk = 0.20 * density

        if gap < safe_gap:
            risk += 1.35
        elif gap < 1.8 * safe_gap:
            risk += 0.55

        if crashed_ahead:
            risk += 2.0

        lane_risks.append(risk)

    target_lane = int(np.argmax(lane_risks))
    target_x = lane_centers[target_lane]
    target_y = max(70.0, leader.y - 210.0)

    uav.x += 0.06 * (target_x - uav.x)
    uav.y += 0.06 * (target_y - uav.y)


def uav_broadcast_to_leader(uav: UAVSupervisor, leader: Vehicle, vehicles: List[Vehicle], frame_idx: int, cfg: TrafficConfig, weather: WeatherProfile):
    dx = leader.x - uav.x
    dy = leader.y - uav.y
    d = math.sqrt(dx * dx + dy * dy)

    if d > uav.coverage:
        return []

    if random.random() > cfg.uav_broadcast_success * weather.comm_factor:
        return []

    safe_gap = weather_safe_gap(cfg, weather)
    messages = []

    for lane in range(cfg.n_lanes):
        density = lane_density_ahead(leader, vehicles, lane, cfg.v2v_lookahead_range)
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < cfg.v2v_lookahead_range
            for v in vehicles
        )

        hazard = 0.15 * density

        if gap < safe_gap:
            hazard += 1.2
        elif gap < 1.8 * safe_gap:
            hazard += 0.55

        if crashed_ahead:
            hazard += 2.2

        if weather.name in ["RAIN", "FOG", "STORM"]:
            hazard += 0.25

        if hazard > 0.12:
            messages.append(
                V2VMessage(
                    sender_id=-1,
                    source="UAV",
                    lane=lane,
                    x=uav.x,
                    y=max(0.0, leader.y - min(gap, cfg.v2v_lookahead_range)),
                    speed=0.0,
                    gap_ahead=gap,
                    density_ahead=density,
                    hazard_level=float(clamp(hazard, 0.0, 4.0)),
                    crashed=crashed_ahead,
                    weather=weather.name,
                    timestamp_frame=frame_idx,
                )
            )

    return messages


# ------------------------------------------------------------
# Leader risk estimation
# ------------------------------------------------------------
def estimate_lane_risk_for_leader(leader: Vehicle, vehicles: List[Vehicle], messages: List[V2VMessage], cfg: TrafficConfig, weather: WeatherProfile):
    risks = [0.0 for _ in range(cfg.n_lanes)]

    sensor_range = weather_sensor_range(cfg, weather)
    safe_gap = weather_safe_gap(cfg, weather)

    # Local sensing
    for lane in range(cfg.n_lanes):
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)
        density = lane_density_ahead(leader, vehicles, lane, sensor_range)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < sensor_range
            for v in vehicles
        )

        risks[lane] += 0.22 * density

        if gap < safe_gap:
            risks[lane] += 1.45
        elif gap < 1.8 * safe_gap:
            risks[lane] += 0.62

        if crashed_ahead:
            risks[lane] += 2.5

    # V2V / UAV messages
    for msg in messages:
        if msg.lane < 0 or msg.lane >= cfg.n_lanes:
            continue

        lane = msg.lane
        gap_to_msg = leader.y - msg.y

        if msg.source == "V2V":
            if gap_to_msg < 0.0 or gap_to_msg > cfg.v2v_lookahead_range:
                continue

            distance_weight = math.exp(-gap_to_msg / cfg.v2v_lookahead_range)
            risks[lane] += distance_weight * msg.hazard_level
            risks[lane] += 0.08 * msg.density_ahead

            if msg.crashed:
                risks[lane] += 2.0

            if msg.speed < 0.65 * cfg.leader_speed_init:
                risks[lane] += 0.45

        elif msg.source == "UAV":
            risks[lane] += 0.90 * msg.hazard_level
            risks[lane] += 0.12 * msg.density_ahead
            if msg.crashed:
                risks[lane] += 2.2

    if weather.name == "RAIN":
        risks = [r + 0.12 for r in risks]
    elif weather.name == "FOG":
        risks = [r + 0.25 for r in risks]
    elif weather.name == "STORM":
        risks = [r + 0.42 for r in risks]

    return risks


# ------------------------------------------------------------
# DQN Leader Agent
# ------------------------------------------------------------
DQN_ACTIONS = [
    "KEEP",
    "SLOW",
    "FAST",
    "LANE_LEFT",
    "LANE_RIGHT",
]

DQN_STATE_DIM = 24
DQN_ACTION_DIM = len(DQN_ACTIONS)


if TORCH_AVAILABLE:
    class DQNNetwork(nn.Module):
        def __init__(self, state_dim: int = DQN_STATE_DIM, action_dim: int = DQN_ACTION_DIM):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, action_dim),
            )

        def forward(self, x):
            return self.net(x)
else:
    DQNNetwork = None


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((
            np.asarray(state, dtype=np.float32),
            int(action),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.asarray(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.asarray(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(self, cfg: TrafficConfig):
        if not TORCH_AVAILABLE:
            raise RuntimeError("PyTorch is not installed. Install it with: pip install torch")

        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.policy_net = DQNNetwork().to(self.device)
        self.target_net = DQNNetwork().to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg.dqn_lr)
        self.replay = ReplayBuffer(cfg.dqn_buffer_size)

        self.epsilon = cfg.dqn_epsilon_start
        self.steps = 0

        if cfg.dqn_load_path and os.path.exists(cfg.dqn_load_path):
            self.policy_net.load_state_dict(torch.load(cfg.dqn_load_path, map_location=self.device))
            self.target_net.load_state_dict(self.policy_net.state_dict())
            print(f"[DQN] Loaded pretrained model: {cfg.dqn_load_path}")

    def select_action(self, state: np.ndarray, training: bool = True) -> int:
        self.steps += 1

        if training and random.random() < self.epsilon:
            return random.randint(0, DQN_ACTION_DIM - 1)

        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.policy_net(s)
            return int(torch.argmax(q, dim=1).item())

    def train_step(self) -> Optional[float]:
        if len(self.replay) < max(self.cfg.dqn_batch_size, self.cfg.dqn_warmup_steps):
            return None

        states, actions, rewards, next_states, dones = self.replay.sample(self.cfg.dqn_batch_size)

        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)

        q_values = self.policy_net(states)
        q_selected = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q = self.target_net(next_states).max(dim=1)[0]
            target = rewards + self.cfg.dqn_gamma * next_q * (1.0 - dones)

        loss = nn.functional.smooth_l1_loss(q_selected, target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 5.0)
        self.optimizer.step()

        self.epsilon = max(self.cfg.dqn_epsilon_min, self.epsilon * self.cfg.dqn_epsilon_decay)

        if self.steps % self.cfg.dqn_target_update_frames == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())

    def save(self):
        if self.cfg.dqn_save_path:
            torch.save(self.policy_net.state_dict(), self.cfg.dqn_save_path)
            print(f"[DQN] Saved model: {self.cfg.dqn_save_path}")


def build_dqn_state(leader: Vehicle, vehicles: List[Vehicle], messages: List[V2VMessage], cfg: TrafficConfig, weather: WeatherProfile) -> np.ndarray:
    risks = estimate_lane_risk_for_leader(leader, vehicles, messages, cfg, weather)

    lane_onehot = [0.0, 0.0, 0.0]
    lane_onehot[int(leader.lane)] = 1.0

    speed_norm = leader.speed / max(1e-6, cfg.leader_speed_max)

    front_gaps = []
    densities = []
    crash_flags = []

    for lane in range(cfg.n_lanes):
        _, gap = nearest_vehicle_ahead(leader, vehicles, lane)

        if gap >= 1e8:
            gap_norm = 1.0
        else:
            gap_norm = clamp(gap / cfg.v2v_lookahead_range, 0.0, 1.0)

        density = lane_density_ahead(leader, vehicles, lane, cfg.v2v_lookahead_range)

        crashed_ahead = any(
            v.alive and v.crashed and v.lane == lane and 0.0 < leader.y - v.y < cfg.v2v_lookahead_range
            for v in vehicles
        )

        front_gaps.append(gap_norm)
        densities.append(clamp(density / 10.0, 0.0, 1.0))
        crash_flags.append(1.0 if crashed_ahead else 0.0)

    v2v_count = sum(1 for m in messages if m.source == "V2V")
    uav_count = sum(1 for m in messages if m.source == "UAV")

    msg_features = [
        clamp(v2v_count / 20.0, 0.0, 1.0),
        clamp(uav_count / 5.0, 0.0, 1.0),
    ]

    weather_features = [
        weather.speed_factor,
        weather.braking_factor,
        weather.visibility_factor,
        weather.comm_factor,
        clamp(weather.crash_factor / 3.0, 0.0, 1.0),
    ]

    cooldown_norm = clamp(leader.lane_cooldown / max(1, cfg.lane_change_cooldown_frames), 0.0, 1.0)
    risk_norm = [clamp(r / 4.0, 0.0, 1.0) for r in risks]

    state = np.array(
        lane_onehot
        + [speed_norm]
        + risk_norm
        + front_gaps
        + densities
        + crash_flags
        + msg_features
        + weather_features
        + [cooldown_norm],
        dtype=np.float32,
    )

    if state.shape[0] != DQN_STATE_DIM:
        raise RuntimeError(f"DQN state has wrong size: {state.shape[0]}")

    return state


def copy_stats_light(stats: SimStats) -> SimStats:
    return SimStats(
        spawned=stats.spawned,
        passed=stats.passed,
        escaped_ahead=stats.escaped_ahead,
        collisions=stats.collisions,
        random_crashes=stats.random_crashes,
        leader_crashes=stats.leader_crashes,
        leader_lane_changes=stats.leader_lane_changes,
        leader_slow_events=stats.leader_slow_events,
        leader_v2v_rx=stats.leader_v2v_rx,
        leader_uav_rx=stats.leader_uav_rx,
        packets_attempted=stats.packets_attempted,
        packets_delivered=stats.packets_delivered,
        min_leader_gap=stats.min_leader_gap,
    )


def compute_dqn_reward(
    leader: Vehicle,
    vehicles: List[Vehicle],
    cfg: TrafficConfig,
    weather: WeatherProfile,
    stats_before: SimStats,
    stats_after: SimStats,
    action_name: str,
    lane_changed: bool,
) -> float:
    reward = 0.0

    max_speed = cfg.leader_speed_max * weather.speed_factor
    reward += 0.12 * clamp(leader.speed / max(1e-6, max_speed), 0.0, 1.0)

    _, front_gap = nearest_vehicle_ahead(leader, vehicles, leader.lane)
    safe_gap = weather_safe_gap(cfg, weather)

    if front_gap >= safe_gap:
        reward += 0.15
    else:
        reward -= 0.45 * (1.0 - clamp(front_gap / max(1e-6, safe_gap), 0.0, 1.0))

    risks = estimate_lane_risk_for_leader(leader, vehicles, [], cfg, weather)
    reward -= 0.10 * clamp(risks[leader.lane], 0.0, 4.0)

    new_leader_crashes = stats_after.leader_crashes - stats_before.leader_crashes
    new_total_crashes = stats_after.collisions - stats_before.collisions

    if new_leader_crashes > 0:
        reward -= 8.0

    if new_total_crashes > 0:
        reward -= 0.8 * new_total_crashes

    if lane_changed:
        reward += 0.10
        reward -= 0.04

    if action_name == "SLOW" and front_gap > 1.8 * safe_gap:
        reward -= 0.05

    return float(clamp(reward, -10.0, 2.0))


# ------------------------------------------------------------
# Hybrid DQN + V2V/UAV leader control
# ------------------------------------------------------------
def control_leader_car(
    leader: Vehicle,
    vehicles: List[Vehicle],
    received_messages: List[V2VMessage],
    lane_centers: List[float],
    cfg: TrafficConfig,
    weather: WeatherProfile,
    stats: SimStats,
    dqn_action: Optional[int] = None,
):
    old_lane = leader.lane

    if leader.lane_cooldown > 0:
        leader.lane_cooldown -= 1

    risks = estimate_lane_risk_for_leader(leader, vehicles, received_messages, cfg, weather)
    stats.risk_history.append(tuple(risks))

    safe_gap = weather_safe_gap(cfg, weather)
    max_decel = cfg.max_decel * weather.braking_factor
    max_speed = cfg.leader_speed_max * weather.speed_factor

    _, front_gap = nearest_vehicle_ahead(leader, vehicles, leader.lane)
    stats.min_leader_gap = min(stats.min_leader_gap, front_gap)

    if cfg.use_dqn_agent and dqn_action is not None:
        action_name = DQN_ACTIONS[int(dqn_action)]
        target_lane = leader.lane

        if action_name == "LANE_LEFT":
            target_lane = leader.lane - 1
        elif action_name == "LANE_RIGHT":
            target_lane = leader.lane + 1

        if action_name in ["LANE_LEFT", "LANE_RIGHT"]:
            valid_lane = 0 <= target_lane < cfg.n_lanes
            cooldown_ok = leader.lane_cooldown == 0

            if valid_lane and cooldown_ok and lane_is_clear_for_leader(leader, vehicles, target_lane, cfg, weather):
                leader.lane = target_lane
                leader.lane_cooldown = cfg.lane_change_cooldown_frames
                stats.leader_lane_changes += 1
            else:
                # Unsafe DQN lane-change request is converted to braking.
                leader.speed = max(cfg.leader_speed_min, leader.speed - max_decel)
                stats.leader_slow_events += 1

        elif action_name == "SLOW":
            leader.speed = max(cfg.leader_speed_min, leader.speed - max_decel)
            stats.leader_slow_events += 1

        elif action_name == "FAST":
            # Safety shield: avoid accelerating into risky or close traffic.
            if front_gap > 1.25 * safe_gap and risks[leader.lane] < cfg.risk_threshold_slowdown:
                leader.speed = min(max_speed, leader.speed + cfg.max_accel)
            else:
                leader.speed = max(cfg.leader_speed_min, leader.speed - 0.5 * max_decel)
                stats.leader_slow_events += 1

        else:
            # KEEP
            if front_gap < safe_gap or risks[leader.lane] > cfg.risk_threshold_slowdown:
                leader.speed = max(cfg.leader_speed_min, leader.speed - max_decel)
                stats.leader_slow_events += 1
            else:
                leader.speed = min(max_speed, leader.speed + 0.5 * cfg.max_accel)

        stats.dqn_action_history.append(action_name)

    else:
        # Rule-based fallback controller using the same V2V/UAV risks.
        current_risk = risks[leader.lane]
        safest_lane = int(np.argmin(risks))
        changed_lane = False

        if (
            current_risk > cfg.risk_threshold_lane_change
            and safest_lane != leader.lane
            and leader.lane_cooldown == 0
            and lane_is_clear_for_leader(leader, vehicles, safest_lane, cfg, weather)
        ):
            leader.lane = safest_lane
            leader.lane_cooldown = cfg.lane_change_cooldown_frames
            stats.leader_lane_changes += 1
            changed_lane = True

        must_slow = front_gap < safe_gap or current_risk > cfg.risk_threshold_slowdown

        if must_slow and not changed_lane:
            leader.speed = max(cfg.leader_speed_min, leader.speed - max_decel)
            stats.leader_slow_events += 1
        else:
            leader.speed = min(max_speed, leader.speed + cfg.max_accel)

    target_x = lane_centers[leader.lane]
    leader.x += cfg.lateral_smoothing * (target_x - leader.x)
    stats.leader_speed_history.append(leader.speed)

    return old_lane != leader.lane


# ------------------------------------------------------------
# Traffic dynamics and crashes
# ------------------------------------------------------------
def update_background_vehicle_speeds(leader: Vehicle, vehicles: List[Vehicle], cfg: TrafficConfig, weather: WeatherProfile):
    all_vehicles = [leader] + vehicles
    safe_gap = weather_safe_gap(cfg, weather)

    for v in vehicles:
        if not v.alive:
            continue

        if v.crashed:
            v.speed = 0.0
            v.crash_timer -= 1
            if v.crash_timer <= 0:
                v.alive = False
            continue

        _, gap = nearest_vehicle_ahead(v, all_vehicles, v.lane)

        in_congestion_zone = v.lane == cfg.congestion_lane and cfg.congestion_y_min <= v.y <= cfg.congestion_y_max

        desired = v.desired_speed
        if in_congestion_zone:
            desired *= cfg.congestion_speed_factor

        if gap < 0.9 * safe_gap:
            v.speed = max(0.45, v.speed - 0.18 * weather.braking_factor)
        else:
            v.speed = min(desired, v.speed + 0.035)


def update_relative_positions(leader: Vehicle, vehicles: List[Vehicle], lane_centers: List[float], cfg: TrafficConfig):
    for v in vehicles:
        if not v.alive:
            continue

        if v.crashed:
            v.y += leader.speed
        else:
            v.y += leader.speed - v.speed

        target_x = lane_centers[v.lane]
        v.x += 0.10 * (target_x - v.x)


def remove_offscreen_vehicles(vehicles: List[Vehicle], cfg: TrafficConfig, stats: SimStats):
    kept = []

    for v in vehicles:
        if not v.alive:
            continue

        if v.y < -240.0:
            stats.escaped_ahead += 1
            continue

        if v.y > cfg.game_h + 220.0:
            stats.passed += 1
            continue

        kept.append(v)

    return kept


def register_crash(stats: SimStats, frame_idx: int, a: Vehicle, b: Optional[Vehicle], reason: str, cfg: TrafficConfig):
    stats.collisions += 1
    if reason == "random":
        stats.random_crashes += 1

    if a.is_leader or (b is not None and b.is_leader):
        stats.leader_crashes += 1

    if not a.is_leader and not a.crashed:
        a.crashed = True
        a.speed = 0.0
        a.crash_timer = cfg.crash_duration_frames

    if b is not None and not b.is_leader and not b.crashed:
        b.crashed = True
        b.speed = 0.0
        b.crash_timer = cfg.crash_duration_frames

    stats.crash_events.append({
        "frame": int(frame_idx),
        "reason": reason,
        "vehicle_a": int(a.vid),
        "vehicle_b": int(b.vid) if b is not None else None,
        "lane": int(a.lane),
        "x": float(a.x),
        "y": float(a.y),
    })


def detect_collisions_and_random_crashes(leader: Vehicle, vehicles: List[Vehicle], cfg: TrafficConfig, weather: WeatherProfile, stats: SimStats, frame_idx: int):
    all_vehicles = [leader] + vehicles

    # Physical collisions. Already-crashed vehicles are treated as known obstacles;
    # repeated collision counting is avoided by skipping crashed participants.
    for i in range(len(all_vehicles)):
        a = all_vehicles[i]
        if not a.alive or a.crashed:
            continue

        ra = a.rect()

        for j in range(i + 1, len(all_vehicles)):
            b = all_vehicles[j]
            if not b.alive or b.crashed:
                continue

            if ra.colliderect(b.rect()):
                register_crash(stats, frame_idx, a, b, "collision", cfg)
                if leader in [a, b]:
                    leader.speed = max(cfg.leader_speed_min, leader.speed * 0.45)

    # Random crashes caused by weather, congestion, and small gaps.
    if not cfg.enable_random_crashes:
        return

    for v in vehicles:
        if not v.alive or v.crashed:
            continue

        _, gap = nearest_vehicle_ahead(v, all_vehicles, v.lane)
        local_density = lane_density_ahead(v, all_vehicles, v.lane, cfg.local_sensor_range)

        in_congestion_zone = v.lane == cfg.congestion_lane and cfg.congestion_y_min <= v.y <= cfg.congestion_y_max

        risk = cfg.base_random_crash_prob
        risk *= weather.crash_factor
        risk *= 1.0 + 0.25 * local_density

        if gap < weather_safe_gap(cfg, weather):
            risk *= 3.0
        if in_congestion_zone:
            risk *= 2.0

        if random.random() < risk:
            register_crash(stats, frame_idx, v, None, "random", cfg)


# ------------------------------------------------------------
# Drawing
# ------------------------------------------------------------
def draw_vehicle(surface: pygame.Surface, v: Vehicle, is_leader=False):
    rect = v.rect()

    if v.sprite is not None:
        img = pygame.transform.smoothscale(v.sprite, (rect.width, rect.height))
        surface.blit(img, rect.topleft)
    else:
        pygame.draw.rect(surface, v.color, rect, border_radius=7)

    if v.crashed:
        pygame.draw.rect(surface, (255, 50, 50), rect, 3, border_radius=7)
        pygame.draw.line(surface, (255, 0, 0), rect.topleft, rect.bottomright, 3)
        pygame.draw.line(surface, (255, 0, 0), rect.topright, rect.bottomleft, 3)
        font = pygame.font.Font(None, 18)
        txt = font.render("CRASH", True, (255, 255, 255))
        surface.blit(txt, (rect.x - 2, rect.y - 18))
        return

    if is_leader:
        pygame.draw.rect(surface, (255, 255, 255), rect, 2, border_radius=7)
        font = pygame.font.Font(None, 18)
        txt = font.render("LEADER", True, (255, 255, 255))
        surface.blit(txt, (rect.x - 8, rect.y - 18))
    else:
        dot = (40, 255, 90) if v.connected else (255, 70, 70)
        pygame.draw.circle(surface, dot, (int(v.x), int(v.y - v.length / 2 - 7)), 4)


def draw_uav(surface: pygame.Surface, uav: UAVSupervisor, cfg: TrafficConfig):
    overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)
    pygame.draw.circle(overlay, (80, 190, 255, 32), (int(uav.x), int(uav.y)), int(uav.coverage))
    surface.blit(overlay, (0, 0))

    pygame.draw.circle(surface, (60, 190, 255), (int(uav.x), int(uav.y)), 11)
    pygame.draw.circle(surface, (255, 255, 255), (int(uav.x), int(uav.y)), 4)

    pygame.draw.line(surface, (255, 255, 255), (int(uav.x - 17), int(uav.y)), (int(uav.x + 17), int(uav.y)), 2)
    pygame.draw.line(surface, (255, 255, 255), (int(uav.x), int(uav.y - 17)), (int(uav.x), int(uav.y + 17)), 2)


def draw_weather_overlay(screen: pygame.Surface, cfg: TrafficConfig, weather: WeatherProfile, frame_idx: int):
    if weather.overlay_alpha <= 0:
        return

    overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)

    if weather.name == "RAIN":
        overlay.fill((40, 70, 90, weather.overlay_alpha))
        for _ in range(45):
            x = random.randint(0, cfg.game_w)
            y = random.randint(0, cfg.game_h)
            pygame.draw.line(overlay, (180, 210, 255, 120), (x, y), (x - 5, y + 14), 1)

    elif weather.name == "FOG":
        overlay.fill((210, 210, 210, weather.overlay_alpha))

    elif weather.name == "STORM":
        overlay.fill((30, 35, 50, weather.overlay_alpha))
        for _ in range(65):
            x = random.randint(0, cfg.game_w)
            y = random.randint(0, cfg.game_h)
            pygame.draw.line(overlay, (160, 190, 255, 135), (x, y), (x - 7, y + 17), 1)
        if frame_idx % 70 < 4:
            overlay.fill((255, 255, 255, 70))

    screen.blit(overlay, (0, 0))


def make_risk_panel(risks: Tuple[float, float, float], cfg: TrafficConfig, panel_w: int, panel_h: int):
    surf = pygame.Surface((panel_w, panel_h))
    surf.fill((18, 18, 28))

    font = pygame.font.Font(None, 21)
    small = pygame.font.Font(None, 18)

    surf.blit(font.render("Leader lane-risk estimate", True, (255, 255, 255)), (10, 10))
    max_r = max(1.0, max(risks))

    for i, risk in enumerate(risks):
        y = 48 + i * 52
        w = int((panel_w - 90) * risk / max_r)

        if risk > cfg.risk_threshold_lane_change:
            color = (255, 80, 80)
        elif risk > cfg.risk_threshold_slowdown:
            color = (255, 190, 70)
        else:
            color = (80, 220, 110)

        surf.blit(small.render(f"Lane {i}", True, (230, 230, 230)), (10, y + 5))
        pygame.draw.rect(surf, (60, 60, 75), (72, y, panel_w - 90, 24))
        pygame.draw.rect(surf, color, (72, y, w, 24))
        surf.blit(small.render(f"{risk:.2f}", True, (255, 255, 255)), (panel_w - 48, y + 4))

    return surf


# ------------------------------------------------------------
# Main simulation
# ------------------------------------------------------------
def run_weather_v2v_traffic_sim(
    cfg: Optional[TrafficConfig] = None,
    seed: int = 12,
    runtime: Optional[object] = None,
    display_result: bool = True,
):
    if cfg is None:
        cfg = TrafficConfig()

    if cfg.use_dqn_agent and not TORCH_AVAILABLE:
        print("[WARNING] PyTorch is not available. Disabling DQN and using rule-based controller.")
        cfg.use_dqn_agent = False
        cfg.dqn_training = False

    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)

    weather = get_weather(cfg)

    pygame.display.init()
    pygame.font.init()

    screen = pygame.display.set_mode((cfg.game_w + cfg.panel_w, cfg.game_h))
    clock = pygame.time.Clock()

    font = pygame.font.Font(None, 22)
    small = pygame.font.Font(None, 18)

    sprites = build_sprite_bank(cfg)
    lane_centers = create_lane_centers(cfg)

    road_x0 = int((cfg.game_w - cfg.road_width) / 2)
    road_rect = pygame.Rect(road_x0, 0, cfg.road_width, cfg.game_h)

    stats = SimStats()
    leader_y = cfg.game_h - 115.0

    leader = Vehicle(
        vid=0,
        lane=1,
        x=float(lane_centers[1]),
        y=leader_y,
        speed=cfg.leader_speed_init * weather.speed_factor,
        desired_speed=cfg.leader_speed_max * weather.speed_factor,
        length=64,
        width=38,
        kind="leader",
        color=(35, 165, 255),
        sprite=sprites["leader"],
        connected=True,
        is_leader=True,
    )

    vehicles: List[Vehicle] = []
    next_id = 1

    for _ in range(cfg.initial_vehicles):
        v, next_id = spawn_vehicle(vehicles, lane_centers, cfg, weather, sprites, stats, next_id, initial=True)
        if v is not None:
            vehicles.append(v)

    uav = UAVSupervisor(x=float(lane_centers[1]), y=leader.y - 220.0, coverage=cfg.uav_coverage)

    dqn_agent = DQNAgent(cfg) if cfg.use_dqn_agent else None

    lane_marker_offset = 0.0
    n_frames = int(cfg.fps * cfg.seconds)

    with imageio.get_writer(cfg.out_gif, mode="I", fps=cfg.fps) as writer:
        for frame_idx in range(n_frames):
            clock.tick(cfg.fps)
            pygame.event.pump()

            if runtime is not None and getattr(runtime, "stop_requested", False):
                print("[CONTROL] Stop requested. Finishing partial GIF export...")
                break

            # --------------------------------------------
            # Statistical arrivals
            # --------------------------------------------
            for lane in range(cfg.n_lanes):
                weather_arrival_factor = 0.88 if weather.name in ["RAIN", "FOG", "STORM"] else 1.0
                arrival_per_frame = cfg.arrival_rate_vpm[lane] * weather_arrival_factor / 60.0 / cfg.fps

                if random.random() < arrival_per_frame:
                    v, next_id = spawn_vehicle(
                        vehicles, lane_centers, cfg, weather, sprites, stats, next_id,
                        initial=False, forced_lane=lane
                    )
                    if v is not None:
                        vehicles.append(v)

            # --------------------------------------------
            # V2V
            # --------------------------------------------
            all_vehicles = [leader] + vehicles
            received = simulate_v2v_transfer(all_vehicles, frame_idx, cfg, weather, stats)
            leader_messages = received.get(leader.vid, [])
            stats.leader_v2v_rx += len([m for m in leader_messages if m.source == "V2V"])

            # --------------------------------------------
            # UAV
            # --------------------------------------------
            if cfg.use_uav_supervisor:
                update_uav(uav, leader, vehicles, lane_centers, cfg, weather)
                uav_msgs = uav_broadcast_to_leader(uav, leader, vehicles, frame_idx, cfg, weather)
                leader_messages.extend(uav_msgs)
                stats.leader_uav_rx += len(uav_msgs)

            # --------------------------------------------
            # DQN + V2V + UAV leader control
            # --------------------------------------------
            stats_before = copy_stats_light(stats)
            dqn_state = None
            dqn_action = None

            if cfg.use_dqn_agent and dqn_agent is not None:
                dqn_state = build_dqn_state(leader, vehicles, leader_messages, cfg, weather)
                dqn_action = dqn_agent.select_action(dqn_state, training=cfg.dqn_training)

            lane_changed = control_leader_car(
                leader=leader,
                vehicles=vehicles,
                received_messages=leader_messages,
                lane_centers=lane_centers,
                cfg=cfg,
                weather=weather,
                stats=stats,
                dqn_action=dqn_action,
            )

            # --------------------------------------------
            # Traffic dynamics
            # --------------------------------------------
            update_background_vehicle_speeds(leader, vehicles, cfg, weather)
            update_relative_positions(leader, vehicles, lane_centers, cfg)
            leader.y = leader_y

            detect_collisions_and_random_crashes(leader, vehicles, cfg, weather, stats, frame_idx)
            vehicles = remove_offscreen_vehicles(vehicles, cfg, stats)

            # --------------------------------------------
            # DQN replay update
            # --------------------------------------------
            if cfg.use_dqn_agent and dqn_agent is not None and dqn_state is not None and dqn_action is not None:
                next_dqn_state = build_dqn_state(leader, vehicles, leader_messages, cfg, weather)
                stats_after = copy_stats_light(stats)
                action_name = DQN_ACTIONS[int(dqn_action)]

                reward = compute_dqn_reward(
                    leader=leader,
                    vehicles=vehicles,
                    cfg=cfg,
                    weather=weather,
                    stats_before=stats_before,
                    stats_after=stats_after,
                    action_name=action_name,
                    lane_changed=lane_changed,
                )

                done = stats_after.leader_crashes > stats_before.leader_crashes
                dqn_agent.replay.push(dqn_state, dqn_action, reward, next_dqn_state, done)
                stats.dqn_reward_history.append(reward)

                if cfg.dqn_training:
                    loss = dqn_agent.train_step()
                    if loss is not None:
                        stats.dqn_loss_history.append(loss)

            # --------------------------------------------
            # Rendering
            # --------------------------------------------
            screen.fill(weather.background_color)
            pygame.draw.rect(screen, (82, 82, 86), road_rect)

            # Congestion overlay
            congestion_rect = pygame.Rect(
                int(road_x0 + cfg.congestion_lane * cfg.road_width / cfg.n_lanes),
                int(cfg.congestion_y_min),
                int(cfg.road_width / cfg.n_lanes),
                int(cfg.congestion_y_max - cfg.congestion_y_min),
            )

            congestion_overlay = pygame.Surface((cfg.game_w, cfg.game_h), pygame.SRCALPHA)
            pygame.draw.rect(congestion_overlay, (255, 120, 30, 55), congestion_rect)
            screen.blit(congestion_overlay, (0, 0))

            # Road borders
            pygame.draw.rect(screen, (255, 220, 40), (road_x0 - 8, 0, 8, cfg.game_h))
            pygame.draw.rect(screen, (255, 220, 40), (road_x0 + cfg.road_width, 0, 8, cfg.game_h))

            # Lane markers
            lane_marker_offset += max(1.0, leader.speed * 2.0)
            if lane_marker_offset > 96:
                lane_marker_offset = 0.0

            for y in range(-120, cfg.game_h + 120, 96):
                yy = y + int(lane_marker_offset)
                for boundary in range(1, cfg.n_lanes):
                    x = int(road_x0 + boundary * cfg.road_width / cfg.n_lanes)
                    pygame.draw.rect(screen, (245, 245, 245), (x - 4, yy, 8, 48))

            # UAV
            if cfg.use_uav_supervisor:
                draw_uav(screen, uav, cfg)

            # V2V / UAV links
            for msg in leader_messages:
                if msg.source == "V2V":
                    color = (80, 255, 140)
                    width = 1
                    target = (int(msg.x), int(msg.y))
                else:
                    color = (80, 190, 255)
                    width = 2
                    target = (int(uav.x), int(uav.y))

                pygame.draw.line(screen, color, (int(leader.x), int(leader.y)), target, width)

            # Vehicles
            for v in vehicles:
                draw_vehicle(screen, v, is_leader=False)
            draw_vehicle(screen, leader, is_leader=True)

            draw_weather_overlay(screen, cfg, weather, frame_idx)

            # --------------------------------------------
            # HUD
            # --------------------------------------------
            current_risks = stats.risk_history[-1] if stats.risk_history else (0.0, 0.0, 0.0)
            connected_count = sum(1 for v in vehicles if v.connected)
            crashed_count = sum(1 for v in vehicles if v.crashed)
            packet_rate = stats.packets_delivered / stats.packets_attempted if stats.packets_attempted > 0 else 0.0

            dqn_action_text = stats.dqn_action_history[-1] if stats.dqn_action_history else "NONE"
            dqn_eps_text = f"{dqn_agent.epsilon:.2f}" if cfg.use_dqn_agent and dqn_agent is not None else "OFF"
            dqn_loss_text = f"{stats.dqn_loss_history[-1]:.3e}" if stats.dqn_loss_history else "N/A"
            dqn_reward_text = f"{stats.dqn_reward_history[-1]:+.3f}" if stats.dqn_reward_history else "N/A"

            if runtime is not None and hasattr(runtime, "update_scope"):
                runtime.update_scope(
                    frame_idx=frame_idx,
                    n_frames=n_frames,
                    weather=weather.name,
                    leader_lane=leader.lane,
                    leader_speed=leader.speed,
                    vehicles=len(vehicles),
                    connected=connected_count,
                    crashed=crashed_count,
                    collisions=stats.collisions,
                    random_crashes=stats.random_crashes,
                    leader_crashes=stats.leader_crashes,
                    leader_lane_changes=stats.leader_lane_changes,
                    leader_slow_events=stats.leader_slow_events,
                    v2v_rx=stats.leader_v2v_rx,
                    uav_rx=stats.leader_uav_rx,
                    pdr=packet_rate,
                    min_gap=stats.min_leader_gap,
                    risks=current_risks,
                    dqn_action=dqn_action_text,
                    dqn_epsilon=dqn_eps_text,
                    dqn_reward=dqn_reward_text,
                    dqn_loss=dqn_loss_text,
                )

            screen.blit(font.render("DQN + V2V/UAV Weather-Aware Traffic Avoidance", True, (255, 255, 255)), (12, 10))
            screen.blit(font.render(f"Weather={weather.name} | Leader lane={leader.lane} | speed={leader.speed:.2f}", True, (255, 255, 255)), (12, 34))
            screen.blit(font.render(f"Vehicles={len(vehicles)} | connected={connected_count} | crashed={crashed_count}", True, (255, 255, 255)), (12, 58))
            screen.blit(font.render(f"V2V RX={stats.leader_v2v_rx} | UAV RX={stats.leader_uav_rx} | PDR={100.0 * packet_rate:.1f}%", True, (255, 255, 255)), (12, 82))
            screen.blit(font.render(f"Lane changes={stats.leader_lane_changes} | slow events={stats.leader_slow_events}", True, (255, 255, 255)), (12, 106))
            screen.blit(font.render(f"Crashes={stats.collisions} | random={stats.random_crashes} | leader={stats.leader_crashes}", True, (255, 255, 255)), (12, 130))
            screen.blit(font.render(f"DQN action={dqn_action_text} | eps={dqn_eps_text} | reward={dqn_reward_text}", True, (255, 255, 255)), (12, 154))
            screen.blit(font.render(f"DQN loss={dqn_loss_text}", True, (255, 255, 255)), (12, 178))

            legend_y = cfg.game_h - 92
            pygame.draw.circle(screen, (40, 255, 90), (20, legend_y), 5)
            screen.blit(small.render("Connected V2V vehicle", True, (255, 255, 255)), (34, legend_y - 7))

            pygame.draw.circle(screen, (255, 70, 70), (20, legend_y + 22), 5)
            screen.blit(small.render("Non-connected vehicle", True, (255, 255, 255)), (34, legend_y + 15))

            pygame.draw.rect(screen, (255, 50, 50), (15, legend_y + 40, 12, 12), 2)
            screen.blit(small.render("Crash / blocked vehicle", True, (255, 255, 255)), (34, legend_y + 37))
            screen.blit(small.render("Orange area: congestion zone", True, (255, 255, 255)), (34, legend_y + 59))

            # Side panel
            panel_x = cfg.game_w
            pygame.draw.rect(screen, (18, 18, 28), (panel_x, 0, cfg.panel_w, cfg.game_h))

            risk_panel = make_risk_panel(current_risks, cfg, cfg.panel_w, 230)
            screen.blit(risk_panel, (panel_x, 0))

            y0 = 250
            screen.blit(font.render("Simulation statistics", True, (255, 255, 255)), (panel_x + 10, y0))

            side_lines = [
                f"Weather mode     : {weather.name}",
                f"Spawned vehicles : {stats.spawned}",
                f"Passed vehicles  : {stats.passed}",
                f"Active vehicles  : {len(vehicles)}",
                f"Crash obstacles  : {crashed_count}",
                f"Packets attempted: {stats.packets_attempted}",
                f"Packets delivered: {stats.packets_delivered}",
                f"Leader V2V RX    : {stats.leader_v2v_rx}",
                f"Leader UAV RX    : {stats.leader_uav_rx}",
                f"Min leader gap   : {stats.min_leader_gap:.1f}px",
                f"DQN enabled      : {cfg.use_dqn_agent}",
                f"DQN action       : {dqn_action_text}",
                f"DQN epsilon      : {dqn_eps_text}",
                f"DQN last reward  : {dqn_reward_text}",
                f"DQN last loss    : {dqn_loss_text}",
            ]

            for i, line in enumerate(side_lines):
                screen.blit(small.render(line, True, (235, 235, 235)), (panel_x + 10, y0 + 34 + 20 * i))

            box_y = cfg.game_h - 138
            pygame.draw.rect(screen, (30, 30, 45), (panel_x + 10, box_y, cfg.panel_w - 20, 124), border_radius=8)

            info = [
                "Hybrid control:",
                "1) V2V/UAV messages form DQN state",
                "2) DQN selects KEEP/SLOW/FAST/lane",
                "3) Safety shield validates action",
                "4) Reward penalizes crash/risk",
                "5) Model saved after simulation",
            ]

            for i, line in enumerate(info):
                screen.blit(small.render(line, True, (255, 255, 255)), (panel_x + 22, box_y + 10 + 20 * i))

            pygame.display.flip()

            arr = pygame.surfarray.array3d(screen)
            arr = np.transpose(arr, (1, 0, 2))
            writer.append_data(arr)

    if cfg.use_dqn_agent and dqn_agent is not None and cfg.dqn_training:
        dqn_agent.save()

    pygame.quit()

    print(f"[DONE] GIF saved: {cfg.out_gif}")
    print(f"[WEATHER] {weather.name}")
    print(f"[STATS] Spawned vehicles     : {stats.spawned}")
    print(f"[STATS] Passed vehicles      : {stats.passed}")
    print(f"[STATS] Total crashes        : {stats.collisions}")
    print(f"[STATS] Random crashes       : {stats.random_crashes}")
    print(f"[STATS] Leader crashes       : {stats.leader_crashes}")
    print(f"[STATS] Leader lane changes  : {stats.leader_lane_changes}")
    print(f"[STATS] Leader slow events   : {stats.leader_slow_events}")
    print(f"[STATS] Packets attempted    : {stats.packets_attempted}")
    print(f"[STATS] Packets delivered    : {stats.packets_delivered}")

    if stats.packets_attempted > 0:
        pdr = 100.0 * stats.packets_delivered / stats.packets_attempted
        print(f"[STATS] Packet delivery rate : {pdr:.2f}%")

    if stats.min_leader_gap < 1e9:
        print(f"[STATS] Minimum leader gap   : {stats.min_leader_gap:.2f}px")

    if stats.dqn_reward_history:
        print(f"[DQN] Last reward            : {stats.dqn_reward_history[-1]:+.4f}")
    if stats.dqn_loss_history:
        print(f"[DQN] Last loss              : {stats.dqn_loss_history[-1]:.4e}")

    if display_result:
        try:
            from IPython.display import Image, display
            display(Image(filename=cfg.out_gif))
        except Exception:
            pass

    return cfg.out_gif, stats


# ------------------------------------------------------------
# Google Colab control panel
# ------------------------------------------------------------
class ColabRuntimeControl:
    """Small runtime object used by the simulation loop and Colab widgets."""

    def __init__(self):
        self.stop_requested = False
        self.thread = None
        self.progress = None
        self.status = None
        self.scope = None
        self.output = None
        self.start_button = None
        self.stop_button = None
        self.reset_button = None

    def request_stop(self):
        self.stop_requested = True
        if self.status is not None:
            self.status.value = "<b>Status:</b> stop requested; current GIF will close safely."

    def reset_scope(self):
        self.stop_requested = False
        if self.progress is not None:
            self.progress.value = 0
        if self.status is not None:
            self.status.value = "<b>Status:</b> ready."
        if self.scope is not None:
            self.scope.value = "<b>Scope:</b> no active simulation."

    def update_scope(self, **m):
        frame_idx = int(m.get("frame_idx", 0))
        n_frames = max(1, int(m.get("n_frames", 1)))
        progress_value = int(100 * (frame_idx + 1) / n_frames)

        if self.progress is not None:
            self.progress.value = min(100, max(0, progress_value))

        risks = m.get("risks", (0.0, 0.0, 0.0))
        risk_text = ", ".join([f"L{i}:{float(r):.2f}" for i, r in enumerate(risks)])
        pdr = 100.0 * float(m.get("pdr", 0.0))
        min_gap = m.get("min_gap", 0.0)
        if min_gap is None or float(min_gap) > 1e8:
            min_gap_text = "N/A"
        else:
            min_gap_text = f"{float(min_gap):.1f}px"

        if self.status is not None:
            self.status.value = (
                f"<b>Status:</b> running | frame {frame_idx + 1}/{n_frames} | "
                f"weather={m.get('weather', 'N/A')} | DQN={m.get('dqn_action', 'NONE')}"
            )

        if self.scope is not None:
            self.scope.value = f"""
            <div style='font-family:monospace; line-height:1.35'>
            <b>Live scope indication</b><br>
            Leader lane: <b>{m.get('leader_lane', 'N/A')}</b> &nbsp;
            speed: <b>{float(m.get('leader_speed', 0.0)):.2f}</b><br>
            Vehicles: {m.get('vehicles', 0)} | connected: {m.get('connected', 0)} | crashed: {m.get('crashed', 0)}<br>
            V2V RX: {m.get('v2v_rx', 0)} | UAV RX: {m.get('uav_rx', 0)} | PDR: {pdr:.1f}%<br>
            Crashes: {m.get('collisions', 0)} | random: {m.get('random_crashes', 0)} | leader: {m.get('leader_crashes', 0)}<br>
            Lane changes: {m.get('leader_lane_changes', 0)} | slow events: {m.get('leader_slow_events', 0)} | min gap: {min_gap_text}<br>
            Lane risk: {risk_text}<br>
            DQN action: <b>{m.get('dqn_action', 'NONE')}</b> | eps: {m.get('dqn_epsilon', 'OFF')} |
            reward: {m.get('dqn_reward', 'N/A')} | loss: {m.get('dqn_loss', 'N/A')}
            </div>
            """


def create_colab_control_panel(default_cfg: Optional[TrafficConfig] = None):
    """
    Create Start, Stop, Reset controls for Google Colab/Jupyter.

    Usage in Colab:
        from weather_v2v_traffic_dqn_v2v_uav_colab import create_colab_control_panel
        create_colab_control_panel()
    """
    try:
        import threading
        import ipywidgets as widgets
        from IPython.display import display, clear_output, Image
    except Exception as exc:
        raise RuntimeError(
            "Colab control panel requires ipywidgets and IPython. Install with: pip install ipywidgets"
        ) from exc

    base_cfg = default_cfg if default_cfg is not None else TrafficConfig()
    control = ColabRuntimeControl()

    weather_dd = widgets.Dropdown(
        options=["clear", "rain", "fog", "storm"],
        value=base_cfg.weather_mode if base_cfg.weather_mode in ["clear", "rain", "fog", "storm"] else "rain",
        description="Weather:",
    )
    seconds_slider = widgets.IntSlider(value=base_cfg.seconds, min=5, max=60, step=1, description="Seconds:")
    fps_slider = widgets.IntSlider(value=base_cfg.fps, min=5, max=30, step=1, description="FPS:")
    seed_int = widgets.IntText(value=12, description="Seed:")
    dqn_check = widgets.Checkbox(value=base_cfg.use_dqn_agent, description="Use DQN")
    train_check = widgets.Checkbox(value=base_cfg.dqn_training, description="Train DQN")
    uav_check = widgets.Checkbox(value=base_cfg.use_uav_supervisor, description="Use UAV")
    gif_text = widgets.Text(value=base_cfg.out_gif, description="GIF:", layout=widgets.Layout(width="420px"))

    start_button = widgets.Button(description="Start simulation", button_style="success", icon="play")
    stop_button = widgets.Button(description="Stop", button_style="warning", icon="stop")
    reset_button = widgets.Button(description="Reset", button_style="danger", icon="refresh")

    progress = widgets.IntProgress(value=0, min=0, max=100, description="Progress:", bar_style="info")
    status = widgets.HTML("<b>Status:</b> ready.")
    scope = widgets.HTML("<b>Scope:</b> no active simulation.")
    output = widgets.Output()

    control.progress = progress
    control.status = status
    control.scope = scope
    control.output = output
    control.start_button = start_button
    control.stop_button = stop_button
    control.reset_button = reset_button

    def make_cfg():
        cfg = TrafficConfig(
            out_gif=gif_text.value,
            weather_mode=weather_dd.value,
            seconds=int(seconds_slider.value),
            fps=int(fps_slider.value),
            use_dqn_agent=bool(dqn_check.value),
            dqn_training=bool(train_check.value),
            use_uav_supervisor=bool(uav_check.value),
            car_img_path=base_cfg.car_img_path,
            taxi_img_path=base_cfg.taxi_img_path,
            semi_trailer_img_path=base_cfg.semi_trailer_img_path,
            pickup_truck_img_path=base_cfg.pickup_truck_img_path,
            initial_vehicles=base_cfg.initial_vehicles,
            max_vehicles=base_cfg.max_vehicles,
            arrival_rate_vpm=base_cfg.arrival_rate_vpm,
            p_connected_vehicle=base_cfg.p_connected_vehicle,
            enable_random_crashes=base_cfg.enable_random_crashes,
            base_random_crash_prob=base_cfg.base_random_crash_prob,
            congestion_lane=base_cfg.congestion_lane,
            congestion_speed_factor=base_cfg.congestion_speed_factor,
            uav_coverage=base_cfg.uav_coverage,
        )
        return cfg

    def run_in_thread():
        cfg = make_cfg()
        control.stop_requested = False
        control.reset_scope()
        start_button.disabled = True
        stop_button.disabled = False
        reset_button.disabled = True
        with output:
            clear_output(wait=True)
            print(f"[COLAB] Starting simulation: weather={cfg.weather_mode}, seconds={cfg.seconds}, fps={cfg.fps}")

        try:
            gif_path, stats = run_weather_v2v_traffic_sim(
                cfg=cfg,
                seed=int(seed_int.value),
                runtime=control,
                display_result=False,
            )
            control.status.value = "<b>Status:</b> completed." if not control.stop_requested else "<b>Status:</b> stopped by user."
            with output:
                print(f"[COLAB] GIF generated: {gif_path}")
                print(f"[COLAB] Crashes={stats.collisions}, Leader crashes={stats.leader_crashes}, V2V RX={stats.leader_v2v_rx}, UAV RX={stats.leader_uav_rx}")
                try:
                    display(Image(filename=gif_path))
                except Exception as exc:
                    print(f"[COLAB] Could not display GIF inline: {exc}")
        except Exception as exc:
            control.status.value = f"<b>Status:</b> error: {exc}"
            with output:
                print(f"[ERROR] {exc}")
        finally:
            start_button.disabled = False
            stop_button.disabled = False
            reset_button.disabled = False

    def on_start(_):
        if control.thread is not None and control.thread.is_alive():
            control.status.value = "<b>Status:</b> simulation already running."
            return
        control.thread = threading.Thread(target=run_in_thread, daemon=True)
        control.thread.start()

    def on_stop(_):
        control.request_stop()

    def on_reset(_):
        control.request_stop()
        control.reset_scope()
        with output:
            clear_output(wait=True)
            print("[COLAB] Reset requested. Press Start simulation to run again.")

    start_button.on_click(on_start)
    stop_button.on_click(on_stop)
    reset_button.on_click(on_reset)

    ui = widgets.VBox([
        widgets.HTML("<h3>V2V/UAV + DQN Leader-Car Simulation Control</h3>"),
        widgets.HBox([weather_dd, seconds_slider, fps_slider, seed_int]),
        widgets.HBox([dqn_check, train_check, uav_check]),
        gif_text,
        widgets.HBox([start_button, stop_button, reset_button]),
        progress,
        status,
        scope,
        output,
    ])

    display(ui)
    return control


# ------------------------------------------------------------
# Demo
# ------------------------------------------------------------
def run_demo():
    cfg = TrafficConfig(
        out_gif="weather_v2v_traffic_dqn_v2v_uav.gif",
        weather_mode="storm",  # try clear, rain, fog, storm

        car_img_path="/home/cbeario/DeepPrior/car.png",
        taxi_img_path="/home/cbeario/DeepPrior/taxi.png",
        semi_trailer_img_path="/home/cbeario/DeepPrior/semi_trailer.png",
        pickup_truck_img_path="/home/cbeario/DeepPrior/pickup_truck.png",

        # DQN hybrid controller
        use_dqn_agent=True,
        dqn_training=True,
        dqn_epsilon_start=0.70,
        dqn_epsilon_min=0.05,
        dqn_epsilon_decay=0.992,
        dqn_warmup_steps=180,
        dqn_target_update_frames=100,

        # Heavier traffic to force decisions
        initial_vehicles=34,
        max_vehicles=54,
        arrival_rate_vpm=(32.0, 62.0, 36.0),

        # More V2V support
        p_connected_vehicle=0.92,

        # Crash and congestion
        enable_random_crashes=True,
        base_random_crash_prob=0.0008,
        congestion_lane=1,
        congestion_speed_factor=0.52,

        # UAV support
        use_uav_supervisor=True,
        uav_coverage=290.0,
    )

    return run_weather_v2v_traffic_sim(cfg=cfg, seed=12)


if __name__ == "__main__":
    run_demo()
